# STAGE-1 PATCH (di atas AUDIT-PATCH v2 / v3)

Perbaikan **Tahap 1 saja**: (1) message diversity, (2) embedder, (3) extractor. Pipeline, codec, payload ID+CRC-16, split dataset, dan audit A0/A1 + kontrol **tidak diganti**.

| # | Sebelum | Sesudah | Sel |
|---|---|---|---|
| 1 | `CLIPS_PER_BATCH=2`, `CLIP_LEN=8` (2 payload x 8 frame). Sel training memakai `repeat_interleave(CLIP_LEN)` | step tanpa ffmpeg: `16 x 1` (16 payload independen/unik, 1 frame/payload), di-`assert` per step; step BPDA/ffmpeg tetap `2 x 8` (`BPDA_*`) | 2 (Config), 6 (helper + training penuh) |
| 2 | `delta = DELTA_MAX * tanh(net(...))` | `delta = net(...)` mentah; loss += `LAMBDA_DELTA * delta.pow(2).mean()`; layer akhir `N(0, 1e-3)`, bias 0 | 5, 6 |
| 3 | conv -> `AdaptiveAvgPool2d((2,2))` -> flatten -> FC | conv -> conv `MSG_BITS=48` kanal -> `mean(dim=(-2,-1))` -> 48 logit | 5 |

**Urutan jalan (gate):** 1-5 -> **sel verifikasi implementasi** -> (hanya bila `IMPLEMENTATION: PASS`) **training diagnostik + A0/A1** -> (hanya bila A0 & A1 > 0.98) Uji B -> training penuh.

**Catatan:** output pada sel lain yang tidak diubah (mis. 25, 27, harness, evaluasi) berasal dari run v3 lama (arsitektur lama) dan **sudah usang**; jalankan ulang setelah retraining. Checkpoint lama tidak kompatibel (path baru bertag `CFG_TAG`).


# AUDIT-PATCH v2

Notebook ini = notebook AUDIT-PATCH sebelumnya + 3 sel tambahan (6b, 6c, 7d). Sel asli **tidak diubah**, output lama dipertahankan. Sel baru diberi judul `[AUDIT v2]`.

Urutan: 1–5 → 6a → **6b (Uji A/B/C, 16 pesan unik)** → **6c (arsip checkpoint)** → 6 (training, hanya bila 6b menunjukkan model bisa belajar) → 7 → 7a → 7b/7c → **7d (gerbang & per-bit)**.

# AUDIT-PATCH — Verifikasi binary payload melalui Latent Neural Codec

Notebook ini adalah **salinan terpatch** dari `watermark_latent_neural_codec.ipynb`. Arsitektur (codec CompressAI, embedder/extractor latent, payload ID+CRC, ECC_REPEAT=1) **tidak diubah**. Semua output lama dikosongkan agar tidak tercampur dengan hasil baru; hasil run lama tetap ada di notebook asli.

**Pertanyaan yang diuji:** apakah *ground-truth bits* pada `watermarked video -> Latent Neural Codec -> decoder` bisa dipulihkan (bukan apakah string tertentu dikenali).

| Sel | Perubahan | Alasan (temuan audit) |
|---|---|---|
| Config (bagian 2) | Validasi dipisah **per grup** dari train; assert tanpa overlap; `EVAL_MAX_RETRY=1`, `STRICT_SANITY`, `DIAG_STEPS` | Val lama bisa satu grup dengan train; retry menutupi first-pass; evaluasi lama memakai alpha percobaan terakhir |
| Bagian 6 (dipecah 2) | `_log_payloads` mencatat semua payload training ke `train_payloads.json`; sanity check jadi **hard stop** (<0.85) | Bukti payload test unseen; run lama tetap lanjut training walau sanity hanya 0.584 |
| **Sel baru** Diagnostik Level 2 | Tangga readout: latent float -> video tanpa rounding -> video + STE round | Melokalisasi di mana informasi hilang **sebelum** training penuh |
| **Sel baru** Freeze | Hash checkpoint terpilih, `requires_grad=False` | Freeze sebelum final test |
| **Sel baru** Harness EXP-01..07 | Set payload (pola/teks/acak/train/val), metrik bit, 4 kondisi readout, gerbang Level 2 di VALIDASI, final test sekali jalan | Aturan Level 1/2/3, unseen payload/video, tanpa tuning di test |
| Bagian 8 | `run_watermark_pipeline(..., max_retry=EVAL_MAX_RETRY)` | Retry OFF untuk eksperimen utama |

**Tidak diubah (diketahui):** baseline DCT-QIM pada bagian 9 masih dikalibrasi memakai frame test; ini hanya memengaruhi baseline, bukan metode usulan.

**Urutan jalan:** bagian 1-5 -> **Diagnostik Level 2** -> training (bagian 6) -> bagian 7 -> **Freeze** -> **Harness** (Level 1 -> gerbang Level 2 di validasi -> final test). Bagian 8-12 asli tetap dipakai sebagai pipeline demo/verifikasi.

# Video Watermarking Tahan Kompresi Tinggi dengan Neural Codec dan Sistem Verifikasi Otomatis

**Notebook Google Colab — versi disesuaikan dengan konsep riset**

Alur algoritma (sesuai konsep):

```
Input Video → Neural Codec Encoder (g_a) → Watermark Embedding di LATENT (y → y_w)
            → Kuantisasi + Entropy Coding (bitstream nyata) → Neural Codec Decoder (g_s) → Video Output
```

Alur sistem verifikasi:

```
Upload video → Ekstraksi watermark (via latent g_a) → Validasi integritas → Laporan otomatis (JSON + HTML)
```

Loop flowchart pembimbing tetap dipertahankan (Bagian 8):

```
Input Video → Insert Watermark → Compress Video (HEVC CRF tinggi) → Rescan Watermark
            → legible? --Tidak--> Insert ulang (alpha dinaikkan, maks MAX_RETRY)
                       --Ya-----> True + payload biner → daftarkan ke registry
```

| Komponen konsep | Bagian notebook |
|---|---|
| Neural codec (encoder/decoder, bitstream nyata) | 4 |
| Penyisipan watermark pada **latent feature** neural codec (Novelty 1) | 5 |
| Pelatihan tahan kompresi modern: H.264, **HEVC**, **AV1**, neural codec (Novelty 2) | 6 |
| Output 1 — algoritma watermarking + loop flowchart | 7–8 |
| Evaluasi: PSNR, SSIM, BER, robustness setelah kompresi, false positive, baseline tradisional | 9 |
| Output 2 — tool verifikasi + validasi integritas + laporan otomatis (Novelty 3) | 10–11 |

**Perbedaan utama dari notebook sebelumnya:** watermark sekarang disisipkan di ruang latent neural codec (bukan di piksel), payload pelatihan 100% acak (mencegah decoder "menghafal" satu teks), payload berisi ID + CRC-16 yang terikat ke registry, pembagian data train/test tanpa kebocoran, evaluasi imperceptibility dan false positive, baseline DCT-QIM, serta sistem verifikasi lengkap.

## 1. Setup Lingkungan

CompressAI dipasang dengan `--no-deps` karena metadata paketnya meminta `numpy<2`, sedangkan Colab memakai numpy 2.x. Fungsi inti CompressAI (model, entropy coder) tetap berjalan normal di numpy 2, jadi runtime **tidak perlu di-restart**.

In [1]:
import sys, subprocess, importlib.util, shutil

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


def _pip(*pkgs, no_deps=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + (['--no-deps'] if no_deps else []) + list(pkgs)
    subprocess.run(cmd, check=False)


if importlib.util.find_spec('compressai') is None:
    _pip('compressai==1.2.8', 'pytorch-msssim', 'einops', no_deps=True)

# compressai 1.2.8 mengimpor compressai.datasets.pointcloud secara unconditional di __init__.py,
# dan modul itu butuh torch_geometric walau kita hanya memakai compressai.zoo.image_models.
# torch_geometric aman diinstall NORMAL (dengan dependensinya sendiri) karena tidak membawa
# pin numpy<2 seperti compressai, jadi tidak akan merusak numpy 2.x yang dipakai Colab.
if importlib.util.find_spec('torch_geometric') is None:
    _pip('torch_geometric')

for mod, pkg in [('pytorch_msssim', 'pytorch-msssim'), ('skimage', 'scikit-image'), ('scipy', 'scipy'),
                 ('cv2', 'opencv-python-headless'), ('pandas', 'pandas'), ('tqdm', 'tqdm')]:
    if importlib.util.find_spec(mod) is None:
        _pip(pkg, no_deps=(mod == 'pytorch_msssim'))

if shutil.which('ffmpeg') is None:
    subprocess.run('apt-get -y install ffmpeg > /dev/null 2>&1', shell=True)

_enc = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
print('ffmpeg ada        :', shutil.which('ffmpeg') is not None)
for name in ['libx264', 'libx265', 'libaom-av1', 'libsvtav1', 'ffv1']:
    print(f'  encoder {name:<11}:', name in _enc)

Mounted at /content/drive
ffmpeg ada        : True
  encoder libx264    : True
  encoder libx265    : True
  encoder libaom-av1 : True
  encoder libsvtav1  : True
  encoder ffv1       : True


## 2. Konfigurasi Global & Pembagian Dataset

Semua parameter ada di satu sel. Set `QUICK_TEST = True` untuk mencoba seluruh alur secara cepat (hasilnya tidak representatif, hanya untuk memastikan kode berjalan).

Pembagian train/test dilakukan **per grup** (misalnya `CricketShot_g01`), karena video UCF101 dalam satu grup berasal dari rekaman yang sama. Kalau dibagi per file, video uji akan sangat mirip dengan video latih dan hasil evaluasi menjadi terlalu optimistis.

In [2]:
import os, re, json, time, hmac, hashlib, random, gc, math, tempfile, datetime, base64, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ------------------------------------------------------------
# LOKASI DATA (sesuaikan)
# ------------------------------------------------------------
DATASET_DIR = '/content/drive/MyDrive/Video_data/test'
OUTPUT_DIR  = '/content/drive/MyDrive/Video_data/output_latent_wm'

QUICK_TEST = False            # True = uji alur cepat (hasil tidak representatif)

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------
FRAME_SIZE           = (192, 256)   # (H, W), WAJIB kelipatan 64 (syarat neural codec hyperprior). 4:3 seperti UCF101
MAX_FRAMES_PER_VIDEO = 100
SEED                 = 42
TEST_RATIO           = 0.3          # proporsi GRUP video untuk uji
N_VAL_VIDEOS         = 4            # diambil dari sisi train, untuk validasi tiap epoch
MAX_TRAIN_VIDEOS     = 60           # dibatasi agar RAM Colab cukup (disimpan uint8)
MAX_EVAL_VIDEOS      = 30           # jumlah video uji untuk pipeline & evaluasi

# ------------------------------------------------------------
# NEURAL CODEC (CompressAI, pretrained)
# ------------------------------------------------------------
CODEC_ARCH            = 'bmshj2018-hyperprior'  # fungsi codec_* di Bagian 4 ditulis khusus arsitektur ini
CODEC_QUALITY         = 3           # 1 = kompresi paling tinggi ... 8 = kualitas paling tinggi
USE_PRETRAINED_CODEC  = True
CODEC_FALLBACK_STEPS  = 3000        # dipakai hanya jika bobot pretrained gagal diunduh

# ------------------------------------------------------------
# PAYLOAD
# ------------------------------------------------------------
OWNER_TEXT = 'sabila'               # identitas pemilik (disimpan di registry, bukan disisipkan mentah)
SECRET_KEY = b'ganti-dengan-kunci-rahasia-anda'   # kunci HMAC untuk membuat ID watermark
ID_BITS    = 32
CRC_BITS   = 16
ECC_REPEAT = 1                      # >1 = repetition code dengan soft-combining

# ------------------------------------------------------------
# EMBEDDING & LOOP FLOWCHART
# ------------------------------------------------------------
DELTA_MAX        = 2.0              # [LEGACY] TIDAK lagi membatasi embedder (tanh dihapus); hanya dipakai untuk pelaporan audit
ALPHA_INIT       = 1.0
MAX_RETRY        = 5
RETRY_ALPHA_STEP = 0.25             # alpha: 1.00, 1.25, 1.50, ...
LOOP_ATTACK      = ('libx265', 35)  # kotak "Compress Video" = HEVC CRF tinggi

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------
EPOCHS           = 40
STEPS_PER_EPOCH  = 150
CLIP_LEN         = 1                # [STAGE-1] 1 frame per payload (step TANPA ffmpeg)
CLIPS_PER_BATCH  = 16               # [STAGE-1] 16 payload UNIK per step (16 klip x 1 frame)
BPDA_CLIP_LEN        = 8            # [STAGE-1] mekanisme sequential-clip LAMA, hanya untuk step BPDA/ffmpeg (tidak diubah)
BPDA_CLIPS_PER_BATCH = 2            # [STAGE-1] idem
LAMBDA_DELTA     = 0.01             # [STAGE-1] bobot penalti L2 delta: LAMBDA_DELTA * delta.pow(2).mean(). Belum di-tune.
STAGE1_DIAG_STEPS = 1500            # [STAGE-1] langkah training diagnostik (rentang customer 1.000-2.000)
LR               = 1e-3
WARMUP_EPOCHS    = 3                # tanpa serangan, fokus agar pesan bisa terbaca dulu
TARGET_PSNR      = 38.0             # anggaran imperceptibility: PSNR(watermarked, codec tanpa watermark)
LAMBDA_IMG_INIT  = 1.0              # bobot citra awal, disesuaikan otomatis tiap epoch
MSE_SCALE        = 100.0
LAMBDA_RATE      = 0.05             # penalti kenaikan bpp akibat watermark
SANITY_STEPS     = 300              # uji cepat: apakah pesan bisa dipelajari tanpa serangan?
REAL_CODEC_PROB  = 0.25             # peluang serangan ffmpeg sungguhan (BPDA) per step
REAL_CODECS      = [('libx264', (23, 40)), ('libx265', (23, 40)), ('av1', (30, 55))]

# ------------------------------------------------------------
# EVALUASI
# ------------------------------------------------------------
EVAL_ATTACKS = [('none', None), ('neural', None),
                ('libx264', 23), ('libx264', 30), ('libx264', 35), ('libx264', 40),
                ('libx265', 23), ('libx265', 30), ('libx265', 35), ('libx265', 40),
                ('av1', 35), ('av1', 45), ('av1', 55),
                ('rescale', 0.5)]
QIM_STEP = 14.0                     # nilai awal baseline DCT-QIM (dikalibrasi otomatis di Bagian 9)

# ------------------------------------------------------------
# VERIFIKASI
# ------------------------------------------------------------
SEGMENT_LEN        = 16             # frame per segmen untuk analisis integritas temporal
SEG_BITACC_THRESH  = 0.80           # segmen dianggap membawa watermark jika akurasi bit >= ini
PHASH_THRESH       = 12             # jarak Hamming pHash (0..64) maksimum untuk frame dianggap sama
DETECT_PVALUE      = 1e-6           # ambang deteksi statistik bila CRC gagal
MAX_VERIFY_FRAMES  = 300
RUN_GRADIO         = True

# ------------------------------------------------------------
# [AUDIT-PATCH] konfigurasi audit
# ------------------------------------------------------------
EVAL_MAX_RETRY = 1      # eksperimen utama: retry OFF (hanya first-pass, alpha=ALPHA_INIT)
STRICT_SANITY  = True   # sanity check < 0.85 -> hentikan (jangan lanjut training penuh)
DIAG_STEPS     = 400    # langkah per mode pada Diagnostik Level 2
CFG_TAG = f'c{CLIPS_PER_BATCH}x{CLIP_LEN}_rawdelta_gap'   # [STAGE-1] checkpoint diberi tag konfigurasi+arsitektur (checkpoint lama TIDAK kompatibel)
NOTEBOOK_VERSION = 'audit_v3+stage1_patch'

if QUICK_TEST:
    FRAME_SIZE = (64, 64); MAX_FRAMES_PER_VIDEO = 24; MAX_TRAIN_VIDEOS = 4; N_VAL_VIDEOS = 1
    MAX_EVAL_VIDEOS = 2; EPOCHS = 2; STEPS_PER_EPOCH = 4; WARMUP_EPOCHS = 1
    CODEC_FALLBACK_STEPS = 20; SANITY_STEPS = 10; MAX_RETRY = 2; SEGMENT_LEN = 8
    EVAL_ATTACKS = [('none', None), ('neural', None), ('libx264', 35), ('libx265', 35), ('av1', 45), ('rescale', 0.5)]

if QUICK_TEST:
    DIAG_STEPS = 10
    STAGE1_DIAG_STEPS = 10

assert FRAME_SIZE[0] % 64 == 0 and FRAME_SIZE[1] % 64 == 0, 'FRAME_SIZE harus kelipatan 64'

for sub in ['', 'watermarked', 'compressed', 'reports', 'demo', 'models']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)
REGISTRY_PATH = os.path.join(OUTPUT_DIR, 'registry.json')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# ------------------------------------------------------------
# Daftar video & pembagian per grup
# ------------------------------------------------------------
_GENERATED_MARKERS = ('_watermarked', '_h264', '_h265', '_av1', '_neuralcodec', '_wm', '_tamper')

def _is_generated_output(filename):
    stem = os.path.splitext(filename)[0].lower()
    return any(m in stem for m in _GENERATED_MARKERS)

all_videos = sorted(f for f in os.listdir(DATASET_DIR)
                    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')) and not _is_generated_output(f))

def group_key(name):
    m = re.match(r'v_(.+?)_g(\d+)_c\d+', name)
    return f'{m.group(1)}_g{m.group(2)}' if m else os.path.splitext(name)[0]

groups = sorted(set(group_key(v) for v in all_videos))
rng_split = random.Random(SEED)
rng_split.shuffle(groups)
n_test_groups = max(1, int(round(len(groups) * TEST_RATIO)))
test_groups = set(groups[:n_test_groups])

train_pool = [v for v in all_videos if group_key(v) not in test_groups]
test_videos = [v for v in all_videos if group_key(v) in test_groups]
rng_split.shuffle(train_pool); rng_split.shuffle(test_videos)
# [AUDIT-PATCH] validasi dipisah PER GRUP dari train (sebelumnya per video -> bisa satu grup dengan train)
_train_groups = sorted(set(group_key(v) for v in train_pool))
random.Random(SEED + 1).shuffle(_train_groups)
val_videos, _val_groups = [], set()
for _g in _train_groups:
    if len(val_videos) >= N_VAL_VIDEOS:
        break
    _val_groups.add(_g); val_videos += [v for v in train_pool if group_key(v) == _g]
train_videos = [v for v in train_pool if group_key(v) not in _val_groups][:MAX_TRAIN_VIDEOS]
eval_videos = test_videos[:MAX_EVAL_VIDEOS]
assert not ({group_key(v) for v in train_videos} & _val_groups), 'KEBOCORAN train-val (grup overlap)'
assert not ({group_key(v) for v in train_videos + val_videos} & test_groups), 'KEBOCORAN train/val-test (grup overlap)'

print(f'Total video asli : {len(all_videos)} ({len(groups)} grup)')
print(f'Train / Val      : {len(train_videos)} / {len(val_videos)} video')
print(f'Test (dipakai)   : {len(eval_videos)} dari {len(test_videos)} video, grup tidak overlap dengan train')

Device: cuda
Total video asli : 225 (35 grup)
Train / Val      : 60 / 7 video
Test (dipakai)   : 30 dari 61 video, grup tidak overlap dengan train


## 3. Utilitas Video (baca, tulis lossless, kompresi H.264 / HEVC / AV1)

Seluruh baca/tulis memakai ffmpeg lewat pipe `rawvideo` supaya konsisten untuk semua codec (termasuk AV1, yang tidak selalu bisa dibaca OpenCV). Video hasil watermark disimpan **lossless (FFV1, RGB)** sehingga satu-satunya kompresi yang terukur adalah kompresi yang memang sedang diuji.

In [3]:
_ENCODERS = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
AV1_ENCODER = 'libsvtav1' if 'libsvtav1' in _ENCODERS else ('libaom-av1' if 'libaom-av1' in _ENCODERS else None)
print('Encoder AV1 yang dipakai:', AV1_ENCODER)


def probe_video(path):
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
           '-show_entries', 'stream=width,height,r_frame_rate,codec_name', '-of', 'json', path]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    st = json.loads(out or '{}').get('streams', [{}])[0]
    num, den = (st.get('r_frame_rate') or '25/1').split('/')
    fps = float(num) / float(den) if float(den) else 25.0
    return {'width': st.get('width'), 'height': st.get('height'),
            'fps': fps if 1 <= fps <= 240 else 25.0, 'codec': st.get('codec_name')}


def read_video(path, frame_size=FRAME_SIZE, max_frames=MAX_FRAMES_PER_VIDEO):
    '''Baca video -> uint8 (T, H, W, 3) RGB yang sudah di-resize ke frame_size.'''
    info = probe_video(path)
    H, W = frame_size
    cmd = ['ffmpeg', '-v', 'error', '-i', path, '-vf', f'scale={W}:{H}:flags=area']
    if max_frames:
        cmd += ['-frames:v', str(max_frames)]
    cmd += ['-f', 'rawvideo', '-pix_fmt', 'rgb24', '-']
    res = subprocess.run(cmd, capture_output=True)
    frames = np.frombuffer(res.stdout, np.uint8)
    n = frames.size // (H * W * 3)
    if n == 0:
        raise RuntimeError(f'Tidak ada frame terbaca dari {path}: {res.stderr.decode(errors="ignore")[-300:]}')
    return frames[:n * H * W * 3].reshape(n, H, W, 3).copy(), info['fps'], (info['width'], info['height'])


def write_video(frames_u8, path, fps, codec='ffv1', crf=None):
    '''Tulis frame uint8 RGB ke file. codec: ffv1 (lossless) | libx264 | libx265 | av1.'''
    T, H, W, _ = frames_u8.shape
    cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24',
           '-s', f'{W}x{H}', '-r', str(fps), '-i', '-']
    if codec == 'ffv1':
        cmd += ['-c:v', 'ffv1', '-pix_fmt', 'gbrp']
    elif codec in ('libx264', 'libx265'):
        cmd += ['-c:v', codec, '-crf', str(crf), '-preset', 'fast', '-pix_fmt', 'yuv420p']
        if codec == 'libx265':
            cmd += ['-x265-params', 'log-level=error']
    elif codec == 'av1':
        if AV1_ENCODER is None:
            raise RuntimeError('ffmpeg tidak memiliki encoder AV1')
        cmd += ['-c:v', AV1_ENCODER, '-crf', str(crf), '-pix_fmt', 'yuv420p']
        cmd += ['-preset', '10'] if AV1_ENCODER == 'libsvtav1' else ['-b:v', '0', '-cpu-used', '8', '-row-mt', '1']
    else:
        raise ValueError(codec)
    cmd += [path]
    res = subprocess.run(cmd, input=np.ascontiguousarray(frames_u8).tobytes(), capture_output=True)
    if res.returncode != 0 or not os.path.exists(path):
        raise RuntimeError(f'ffmpeg gagal menulis {path}: {res.stderr.decode(errors="ignore")[-400:]}')
    return path


def codec_ext(codec):
    return {'ffv1': '.mkv', 'libx264': '.mp4', 'libx265': '.mp4', 'av1': '.mkv'}[codec]


def ffmpeg_roundtrip(frames_u8, fps, codec, crf):
    '''Encode lalu decode ulang -> (frame hasil, ukuran file byte).'''
    with tempfile.TemporaryDirectory() as td:
        p = os.path.join(td, 'x' + codec_ext(codec))
        write_video(frames_u8, p, fps, codec=codec, crf=crf)
        size = os.path.getsize(p)
        out, _, _ = read_video(p, frame_size=frames_u8.shape[1:3], max_frames=None)
    if len(out) < len(frames_u8):
        out = np.concatenate([out, np.repeat(out[-1:], len(frames_u8) - len(out), 0)])
    return out[:len(frames_u8)], size


def to_tensor(frames_u8):
    return torch.from_numpy(frames_u8).permute(0, 3, 1, 2).float().div(255.0)


def to_uint8(x):
    return (x.clamp(0, 1).mul(255.0).round().byte().permute(0, 2, 3, 1).cpu().numpy())


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


# --- Cache frame train/val (uint8 agar hemat RAM) ---
train_cache, val_cache = {}, {}
for v in tqdm(train_videos, desc='Load train'):
    train_cache[v] = read_video(os.path.join(DATASET_DIR, v))[0]
for v in val_videos:
    val_cache[v] = read_video(os.path.join(DATASET_DIR, v))[0]
_ram = sum(a.nbytes for a in train_cache.values()) / 1e9
print(f'Train cache: {len(train_cache)} video, {sum(len(a) for a in train_cache.values())} frame, {_ram:.2f} GB')

Encoder AV1 yang dipakai: libsvtav1


Load train:   0%|          | 0/60 [00:00<?, ?it/s]

Train cache: 60 video, 5648 frame, 0.83 GB


## 4a. Payload: ID (HMAC) + CRC-16 + ECC

Payload tidak lagi berupa teks mentah `"sabila"`. Susunannya:

| Bagian | Bit | Fungsi |
|---|---|---|
| Watermark ID | 32 | `HMAC-SHA256(SECRET_KEY, owner‖video‖waktu)`, dipotong 32 bit. Tidak bisa dipalsukan tanpa kunci; menunjuk ke catatan di registry (pemilik, hash konten, dst.) |
| CRC-16/CCITT | 16 | Memastikan payload yang terbaca **valid**. Payload acak lolos CRC dengan peluang 2⁻¹⁶ ≈ 1,5×10⁻⁵ |

Keuntungannya: (1) identitas pemilik tetap tersimpan lengkap di registry, (2) keputusan "legible" punya dasar statistik yang jelas, (3) video tanpa watermark hampir mustahil dinyatakan "legible".

Karena satu payload yang sama disisipkan di **semua frame**, ekstraksi memakai *soft voting* (rata-rata logit antar frame), yang berfungsi sebagai ECC temporal yang sangat kuat.

In [4]:
from scipy.stats import binom

PAYLOAD_BITS = ID_BITS + CRC_BITS
MSG_BITS = PAYLOAD_BITS * ECC_REPEAT


def int_to_bits(v, n):
    return [(v >> (n - 1 - i)) & 1 for i in range(n)]


def bits_to_int(bits):
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return v


def crc16_ccitt(data_bytes, poly=0x1021, init=0xFFFF):
    crc = init
    for byte in data_bytes:
        crc ^= byte << 8
        for _ in range(8):
            crc = ((crc << 1) ^ poly) if (crc & 0x8000) else (crc << 1)
            crc &= 0xFFFF
    return crc


def make_watermark_id(owner, video_name, timestamp):
    msg = f'{owner}|{video_name}|{timestamp}'.encode()
    return int.from_bytes(hmac.new(SECRET_KEY, msg, hashlib.sha256).digest()[:ID_BITS // 8], 'big')


def build_message(wm_id):
    '''ID -> [ID | CRC16] -> diulang ECC_REPEAT kali -> np.float32 (MSG_BITS,)'''
    crc = crc16_ccitt(wm_id.to_bytes(ID_BITS // 8, 'big'))
    payload = int_to_bits(wm_id, ID_BITS) + int_to_bits(crc, CRC_BITS)
    return np.array(payload * ECC_REPEAT, dtype=np.float32)


def decode_message(mean_logits):
    '''Soft-combining antar salinan ECC, lalu cek CRC.'''
    l = np.asarray(mean_logits, dtype=np.float64).reshape(ECC_REPEAT, PAYLOAD_BITS).mean(0)
    bits = (l > 0).astype(int)
    wm_id = bits_to_int(bits[:ID_BITS])
    crc_read = bits_to_int(bits[ID_BITS:])
    crc_ok = crc16_ccitt(wm_id.to_bytes(ID_BITS // 8, 'big')) == crc_read
    prob = 1 / (1 + np.exp(-l))
    confidence = float(np.mean(np.abs(prob - 0.5) * 2))
    return {'bits': bits, 'id': wm_id, 'id_hex': f'{wm_id:08x}', 'crc_ok': bool(crc_ok),
            'confidence': confidence, 'soft': l}


def match_pvalue(n_match, n_total):
    '''P(>= n_match bit cocok secara kebetulan) jika video TIDAK mengandung watermark tsb.'''
    return float(binom.sf(n_match - 1, n_total, 0.5))


def bit_error_rate(a, b):
    return float(np.mean(np.asarray(a).astype(int).ravel() != np.asarray(b).astype(int).ravel()))


_demo_id = make_watermark_id(OWNER_TEXT, 'contoh.mp4', '2026-01-01T00:00:00')
_demo_msg = build_message(_demo_id)
_dec = decode_message((_demo_msg * 2 - 1) * 5)
assert _dec['id'] == _demo_id and _dec['crc_ok']
print(f'PAYLOAD_BITS={PAYLOAD_BITS}, ECC_REPEAT={ECC_REPEAT} -> MSG_BITS={MSG_BITS}')
print(f'Contoh ID: {_demo_id:08x} | payload: {"".join(map(str, _demo_msg.astype(int)))} | CRC ok: {_dec["crc_ok"]}')

PAYLOAD_BITS=48, ECC_REPEAT=1 -> MSG_BITS=48
Contoh ID: e18081b0 | payload: 111000011000000010000001101100001100001010110110 | CRC ok: True


## 4b. Neural Codec (CompressAI `bmshj2018-hyperprior`, pretrained)

Codec lama pada notebook sebelumnya (autoencoder 8 kanal, dilatih 300 frame) hanya mencapai recon loss ≈0,017–0,033 (PSNR sekitar 15–18 dB) dan tidak menghitung bitrate, sehingga belum layak disebut *codec*. Di sini dipakai **learned image codec** yang sudah dilatih pada jutaan citra (Ballé dkk., 2018), dengan komponen:

- `g_a` — analysis transform (**Neural Codec Encoder**): frame → latent `y`
- kuantisasi `round(y)` + entropy coding dengan hyperprior → **bitstream nyata** (bpp terukur)
- `g_s` — synthesis transform (**Neural Codec Decoder**): latent → frame

Codec dijalankan per frame (intra). Semua bobot codec **dibekukan**; yang dilatih hanya modul watermark. Jika unduhan bobot pretrained gagal, codec dilatih ulang dengan loss rate–distortion sebagai cadangan.

In [5]:
from compressai.zoo import image_models
from pytorch_msssim import ssim as ssim_torch

CODEC_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, 'models', f'codec_{CODEC_ARCH}_q{CODEC_QUALITY}_fallback.pth')
_RD_LAMBDA = {1: 0.0018, 2: 0.0035, 3: 0.0067, 4: 0.0130, 5: 0.0250, 6: 0.0483, 7: 0.0932, 8: 0.1800}


def load_codec():
    try:
        m = image_models[CODEC_ARCH](quality=CODEC_QUALITY, pretrained=USE_PRETRAINED_CODEC)
        return m.to(device), USE_PRETRAINED_CODEC
    except Exception as e:
        print('[PERINGATAN] Bobot pretrained gagal dimuat:', repr(e)[:200])
        return image_models[CODEC_ARCH](quality=CODEC_QUALITY, pretrained=False).to(device), False


CODEC_FALLBACK_CKPT_PATH = os.path.join(OUTPUT_DIR, 'models', f'codec_{CODEC_ARCH}_q{CODEC_QUALITY}_fallback_ckpt.pth')
CODEC_FALLBACK_CKPT_EVERY = 100   # simpan progres tiap 100 step, supaya tidak hilang total kalau sesi putus


def train_codec_fallback(model, steps):
    '''Cadangan: latih codec dengan loss rate-distortion (lambda * 255^2 * MSE + bpp).
    Checkpoint/resume per-step ditambahkan supaya training tidak perlu diulang dari 0 kalau
    sesi Colab terputus di tengah jalan (jalur ini hanya aktif kalau bobot pretrained gagal diunduh).'''
    lmbda = _RD_LAMBDA[CODEC_QUALITY]
    main_p = [p for n, p in model.named_parameters() if not n.endswith('.quantiles')]
    aux_p = [p for n, p in model.named_parameters() if n.endswith('.quantiles')]
    opt, aux_opt = torch.optim.Adam(main_p, lr=1e-4), torch.optim.Adam(aux_p, lr=1e-3)
    model.train()
    frames_all = [a for a in train_cache.values()]

    start_step = 0
    if os.path.exists(CODEC_FALLBACK_CKPT_PATH):
        ck = torch.load(CODEC_FALLBACK_CKPT_PATH, map_location=device, weights_only=False)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt']); aux_opt.load_state_dict(ck['aux_opt'])
        start_step = ck['step']
        print(f'[RESUME] Codec fallback melanjutkan dari step {start_step}/{steps}.')

    for step in tqdm(range(start_step, steps), desc='Codec fallback', initial=start_step, total=steps):
        clips = [random.choice(frames_all) for _ in range(8)]
        batch = np.stack([a[random.randrange(len(a))] for a in clips])
        x = to_tensor(batch).to(device)
        out = model(x)
        num_px = x.size(0) * x.size(2) * x.size(3)
        bpp = sum(torch.log(l).sum() for l in out['likelihoods'].values()) / (-math.log(2) * num_px)
        loss = lmbda * 255 ** 2 * F.mse_loss(out['x_hat'], x) + bpp
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(main_p, 1.0); opt.step()
        aux = model.aux_loss(); aux_opt.zero_grad(); aux.backward(); aux_opt.step()
        if (step + 1) % max(1, steps // 10) == 0:
            print(f'  step {step+1}: loss={loss.item():.4f} bpp={bpp.item():.3f}')
        if (step + 1) % CODEC_FALLBACK_CKPT_EVERY == 0 or (step + 1) == steps:
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'aux_opt': aux_opt.state_dict(), 'step': step + 1}, CODEC_FALLBACK_CKPT_PATH)
    torch.save(model.state_dict(), CODEC_WEIGHTS_PATH)
    if os.path.exists(CODEC_FALLBACK_CKPT_PATH):
        os.remove(CODEC_FALLBACK_CKPT_PATH)  # sudah final, checkpoint parsial tidak perlu lagi


codec, CODEC_PRETRAINED = load_codec()
if not CODEC_PRETRAINED:
    if os.path.exists(CODEC_WEIGHTS_PATH):
        codec.load_state_dict(torch.load(CODEC_WEIGHTS_PATH, map_location=device))
        print('Bobot codec cadangan dimuat dari', CODEC_WEIGHTS_PATH)
    else:
        train_codec_fallback(codec, CODEC_FALLBACK_STEPS)

codec.eval()
for p in codec.parameters():
    p.requires_grad_(False)
codec.update(force=True)

with torch.no_grad():
    _y = codec.g_a(torch.zeros(1, 3, *FRAME_SIZE, device=device))
LATENT_CH, LATENT_H, LATENT_W = _y.shape[1:]
with torch.no_grad():   # skala latent -> normalisasi input embedder/extractor (tidak bergantung pada codec)
    LATENT_SCALE = float(codec.g_a(to_tensor(next(iter(val_cache.values()))[:8]).to(device)).std().clamp_min(1e-3))
print(f'Skala latent (std): {LATENT_SCALE:.3f}')
print(f'Neural codec: {CODEC_ARCH} q={CODEC_QUALITY} | pretrained={CODEC_PRETRAINED} | '
      f'latent {LATENT_CH}x{LATENT_H}x{LATENT_W} untuk frame {FRAME_SIZE}')


# ---------------- Fungsi-fungsi codec ----------------
def ste_round(y):
    return y + (torch.round(y) - y).detach()


def codec_roundtrip(x, differentiable=False):
    '''Kompresi neural codec penuh: g_s(Q(g_a(x))).'''
    y = codec.g_a(x)
    y_hat = ste_round(y) if differentiable else torch.round(y)
    return codec.g_s(y_hat).clamp(0, 1)


def latent_rate_bits(y):
    '''Estimasi bit (diferensiabel) untuk latent y dengan entropy model hyperprior.'''
    z = codec.h_a(torch.abs(y))
    z_hat, z_lik = codec.entropy_bottleneck(z, training=True)
    scales = codec.h_s(z_hat)
    _, y_lik = codec.gaussian_conditional(y, scales, training=True)
    return (-torch.log2(y_lik)).sum(dim=(1, 2, 3)) + (-torch.log2(z_lik)).sum(dim=(1, 2, 3))


@torch.no_grad()
def codec_code_latent(y):
    '''Entropy coding NYATA untuk latent y -> (y_hat hasil decode bitstream, byte per frame).'''
    z = codec.h_a(torch.abs(y))
    z_strings = codec.entropy_bottleneck.compress(z)
    z_hat = codec.entropy_bottleneck.decompress(z_strings, z.size()[-2:])
    scales = codec.h_s(z_hat)
    idx = codec.gaussian_conditional.build_indexes(scales)
    y_strings = codec.gaussian_conditional.compress(y, idx)
    y_hat = codec.gaussian_conditional.decompress(y_strings, idx, z_hat.dtype)
    nbytes = [len(a) + len(b) for a, b in zip(y_strings, z_strings)]
    return y_hat, nbytes


# Kualitas codec pada frame validasi (referensi)
with torch.no_grad():
    _v = to_tensor(next(iter(val_cache.values()))[:8]).to(device)
    _yh, _nb = codec_code_latent(codec.g_a(_v))
    _xr = codec.g_s(_yh).clamp(0, 1)
    _psnr = (10 * torch.log10(1 / F.mse_loss(_xr, _v))).item()
print(f'Codec tanpa watermark: PSNR={_psnr:.2f} dB | bpp={np.mean(_nb) * 8 / (FRAME_SIZE[0] * FRAME_SIZE[1]):.3f}')

Downloading: "https://compressai.s3.amazonaws.com/models/v1/bmshj2018-hyperprior-3-6d87be32.pth.tar" to /root/.cache/torch/hub/checkpoints/bmshj2018-hyperprior-3-6d87be32.pth.tar


100%|██████████| 20.2M/20.2M [00:02<00:00, 8.54MB/s]


Skala latent (std): 1.537
Neural codec: bmshj2018-hyperprior q=3 | pretrained=True | latent 192x12x16 untuk frame (192, 256)
Codec tanpa watermark: PSNR=32.43 dB | bpp=0.278


## 5. Arsitektur Watermark di Ruang Latent (Novelty 1)

**LatentWatermarkEmbedder** — menerima latent `y = g_a(x)` dan pesan `m`, menghasilkan perubahan latent terbatas:

$$y_w = y + \alpha \cdot \Delta_{max} \cdot \tanh(f_\theta(y, m))$$

Perubahan dibuat *content-adaptive* (bergantung pada `y`), dibatasi `DELTA_MAX` agar imperceptible, dan `α` dapat dinaikkan pada loop retry flowchart. Setelah itu `y_w` dikuantisasi, di-entropy-coding, dan didekode `g_s` → video ber-watermark.

**LatentWatermarkExtractor** — video yang diterima (bisa sudah dikompresi HEVC/AV1) **dipetakan kembali ke ruang latent** oleh `g_a` yang sama, lalu jaringan kecil membaca pesan dari latent tersebut. Inilah bentuk konkret hipotesis "ruang fitur neural codec lebih stabil terhadap transformasi": penyisipan maupun pembacaan terjadi di ruang latent yang sama.

In [6]:
def _gn(c):
    return nn.GroupNorm(8 if c % 8 == 0 else 1, c)


class LatentWatermarkEmbedder(nn.Module):
    '''[STAGE-1] delta = keluaran MENTAH jaringan (tanpa tanh/sigmoid/clamp); y_w = y + alpha * delta.
    Besar delta dibatasi oleh penalti L2 di loss (LAMBDA_DELTA * delta.pow(2).mean()), bukan oleh saturasi.'''
    def __init__(self, msg_bits=MSG_BITS, latent_ch=LATENT_CH, hidden=192, msg_ch=32):
        super().__init__()
        self.msg_ch = msg_ch
        self.msg_fc = nn.Sequential(nn.Linear(msg_bits, 256), nn.ReLU(inplace=True),
                                    nn.Linear(256, msg_ch * 4 * 4))
        self.net = nn.Sequential(
            nn.Conv2d(latent_ch + msg_ch, hidden, 3, padding=1), _gn(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1), _gn(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1), _gn(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, latent_ch, 1))
        self._init_final_layer()

    def final_layer(self):
        '''Layer terakhir = penghasil delta (dicari dari struktur, bukan dari nama).'''
        return [m for m in self.net.modules() if isinstance(m, nn.Conv2d)][-1]

    def _init_final_layer(self):
        last = self.final_layer()
        nn.init.normal_(last.weight, mean=0.0, std=1e-3)
        nn.init.zeros_(last.bias)

    def forward(self, y, msg, alpha=1.0):
        B = y.size(0)
        m = self.msg_fc(msg * 2 - 1).view(B, self.msg_ch, 4, 4)
        m = F.interpolate(m, size=y.shape[-2:], mode='bilinear', align_corners=False)
        delta = self.net(torch.cat([y / LATENT_SCALE, m], dim=1))      # raw output layer terakhir
        return y + alpha * delta, delta


class LatentWatermarkExtractor(nn.Module):
    '''[STAGE-1] HiDDeN-style: Conv -> ... -> Conv(MSG_BITS kanal) -> Global Average Pooling -> MSG_BITS logits.
    Tidak ada AdaptiveAvgPool2d((2,2)) + FC. Keluaran = logit mentah (untuk BCEWithLogitsLoss), bukan probabilitas.
    forward(x): video -> g_a (beku) -> latent -> logits.  read_latent(y): latent langsung -> logits (tanpa g_a).'''
    def __init__(self, msg_bits=MSG_BITS, latent_ch=LATENT_CH, hidden=256):
        super().__init__()
        self.msg_bits = msg_bits
        self.net = nn.Sequential(
            nn.Conv2d(latent_ch, hidden, 3, padding=1), _gn(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1), _gn(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, hidden, 3, padding=1), _gn(hidden), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, msg_bits, 3, padding=1))                 # -> [B, MSG_BITS, H, W]

    def head(self, y_scaled):
        feats = self.net(y_scaled)                                     # [B, MSG_BITS, H, W]
        return feats.mean(dim=(-2, -1))                                # GAP -> [B, MSG_BITS] logits

    def read_latent(self, y):
        return self.head(y / LATENT_SCALE)

    def forward(self, x):
        return self.head(codec.g_a(x) / LATENT_SCALE)


embedder = LatentWatermarkEmbedder().to(device)
extractor = LatentWatermarkExtractor().to(device)
n_par = sum(p.numel() for p in list(embedder.parameters()) + list(extractor.parameters()))
print(f'Parameter watermark yang dilatih: {n_par/1e6:.2f} M (codec beku tidak ikut dilatih)')


Parameter watermark yang dilatih: 2.97 M (codec beku tidak ikut dilatih)


## 6. Pelatihan Tahan Kompresi Modern (Novelty 2)

Perbaikan penting dibanding versi lama:

1. **Pesan 100% acak** di setiap klip. Pada versi lama 50% batch memakai teks target `"sabila"`, sehingga decoder cukup menghafal satu jawaban. Akurasi training yang tertahan di ≈0,75 sepanjang 200 epoch adalah tanda khasnya: `0,5 × 1,0 (batch target) + 0,5 × 0,5 (batch acak, ditebak)` = 0,75.
2. **Noise layer** mencakup gaussian, blur, rescale, perubahan warna, neural codec (STE), dan **H.264 / HEVC / AV1 sungguhan** via BPDA (forward = ffmpeg, backward = identity).
3. **Klip frame berurutan** dari video yang sama, sehingga codec video bekerja realistis (prediksi antar-frame).
4. **Loss** = BCE pesan + λ_img·(MSE·100 + (1−SSIM)) terhadap output codec *tanpa* watermark + λ_rate·kenaikan bpp.
5. **Anggaran imperceptibility otomatis**: λ_img dinaikkan bila PSNR validasi di bawah `TARGET_PSNR`, diturunkan bila di atasnya.
6. **Sanity check** sebelum training penuh: tanpa serangan, pesan acak harus bisa dipelajari (bit_acc > 0,85). Jika gagal, notebook memberi peringatan sebelum Anda menghabiskan jam GPU.
7. **Validasi** tiap epoch pada video terpisah, termasuk HEVC CRF 35 sungguhan; checkpoint terbaik dipilih dari skor validasi, bukan dari akurasi training.

In [7]:
def psnr_t(a, b):
    return (10 * torch.log10(1.0 / F.mse_loss(a, b).clamp_min(1e-10))).item()


def _gauss_kernel(sigma, k=5):
    ax = torch.arange(k, dtype=torch.float32, device=device) - k // 2
    g = torch.exp(-ax ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    return (g[:, None] * g[None, :]).expand(3, 1, k, k).contiguous()


def sample_clips(cache, n_clips=CLIPS_PER_BATCH, clip_len=CLIP_LEN):
    clips = []
    keys = list(cache.keys())
    for _ in range(n_clips):
        a = cache[random.choice(keys)]
        s = random.randint(0, max(len(a) - clip_len, 0))
        c = a[s:s + clip_len]
        if len(c) < clip_len:
            c = np.concatenate([c, np.repeat(c[-1:], clip_len - len(c), 0)])
        clips.append(c)
    return np.concatenate(clips)             # (n_clips*clip_len, H, W, 3)


# ---------------- [STAGE-1] Message diversity + loss pesan/delta ----------------
def unique_messages(n_msg, bits=None):
    '''n_msg payload acak INDEPENDEN (48 bit murni), dijamin unik dalam satu step.'''
    bits = bits or MSG_BITS
    for _ in range(100):
        m = torch.randint(0, 2, (n_msg, bits), device=device)
        if len({tuple(r) for r in m.int().tolist()}) == n_msg:
            return m.float()
    raise RuntimeError('gagal membuat pesan unik')


def payload_diversity(msg_f):
    '''Ukur payload yang BENAR-BENAR masuk ke model: (jumlah payload unik, frame terbanyak per payload).'''
    counts = {}
    for r in msg_f.detach().int().cpu().tolist():
        counts[tuple(r)] = counts.get(tuple(r), 0) + 1
    return len(counts), max(counts.values())


def make_step_messages(n_clips, clip_len):
    '''1 payload per klip; tiap payload dipakai clip_len frame. Step tanpa ffmpeg: clip_len=1 -> tiap frame payload sendiri.'''
    msgs = unique_messages(n_clips)
    msg_f = msgs.repeat_interleave(clip_len, 0)
    n_uniq, max_fp = payload_diversity(msg_f)
    assert n_uniq == n_clips and max_fp == clip_len, f'diversity rusak: unik={n_uniq} maxfp={max_fp}'
    return msgs, msg_f


def delta_l2(delta):
    return delta.pow(2).mean()


def msg_delta_loss(logits, msg, delta):
    '''bit_loss (BCEWithLogits; extractor mengeluarkan logit) + LAMBDA_DELTA * delta L2. Return (bit, delta_l2, total).'''
    bit = F.binary_cross_entropy_with_logits(logits, msg)
    dl = delta_l2(delta)
    return bit, dl, bit + LAMBDA_DELTA * dl


def bpda_real_codec(x, n_clips, codec_name, crf):
    '''Forward: kompresi ffmpeg sungguhan per klip. Backward: identity (BPDA).'''
    frames = to_uint8(x.detach())
    out = []
    for c in np.split(frames, n_clips):
        out.append(ffmpeg_roundtrip(c, 25, codec_name, crf)[0])
    xc = to_tensor(np.concatenate(out)).to(x.device)
    return x + (xc - x).detach()


def attack_layer(x, n_clips, epoch, use_real=None):
    '''use_real=None: acak (perilaku lama); True/False: diputuskan pemanggil agar ukuran klip sesuai (BPDA vs non-BPDA).'''
    if epoch <= WARMUP_EPOCHS:
        return x, 'none'
    if use_real is None:
        use_real = random.random() < REAL_CODEC_PROB
    if use_real:
        options = [c for c in REAL_CODECS if not (c[0] == 'av1' and AV1_ENCODER is None)]
        name, (lo, hi) = random.choice(options)
        return bpda_real_codec(x, n_clips, name, random.randint(lo, hi)), name
    mode = random.choice(['identity', 'gauss', 'blur', 'rescale', 'color', 'neural', 'neural'])
    if mode == 'gauss':
        x = x + torch.randn_like(x) * random.uniform(0.005, 0.03)
    elif mode == 'blur':
        x = F.conv2d(F.pad(x, (2, 2, 2, 2), mode='replicate'), _gauss_kernel(random.uniform(0.5, 1.2)), groups=3)
    elif mode == 'rescale':
        s = random.uniform(0.5, 0.9); H, W = x.shape[-2:]
        x = F.interpolate(F.interpolate(x, scale_factor=s, mode='bilinear', align_corners=False),
                          size=(H, W), mode='bilinear', align_corners=False)
    elif mode == 'color':
        x = (x - 0.5) * random.uniform(0.8, 1.2) + 0.5 + random.uniform(-0.08, 0.08)
    elif mode == 'neural':
        x = codec_roundtrip(x, differentiable=True)
    return x.clamp(0, 1), mode


def embed_tensor(x, msg, alpha=1.0, differentiable=True):
    with torch.no_grad():
        y = codec.g_a(x)
    y_w, delta = embedder(y, msg, alpha)
    y_w_hat = ste_round(y_w) if differentiable else torch.round(y_w)
    return codec.g_s(y_w_hat).clamp(0, 1), y, y_w


@torch.no_grad()
def validate(n_frames=16):
    embedder.eval(); extractor.eval()
    res = {'psnr_codec': [], 'psnr_orig': [], 'ssim_codec': []}
    attacks = [('none', None), ('neural', None), ('libx265', 35)] + ([('av1', 45)] if AV1_ENCODER else [])
    acc = {f'{a}{"" if c is None else c}': [] for a, c in attacks}
    vacc = {k: [] for k in acc}
    rng = np.random.RandomState(0)
    for v, frames in val_cache.items():
        f = frames[:n_frames]
        x = to_tensor(f).to(device)
        msg = torch.from_numpy(rng.randint(0, 2, MSG_BITS).astype(np.float32)).to(device)
        msgs = msg.expand(len(f), -1)
        x_w, y, _ = embed_tensor(x, msgs, differentiable=False)
        x_c = codec.g_s(torch.round(y)).clamp(0, 1)
        x_w_u8 = to_uint8(x_w)
        x_w = to_tensor(x_w_u8).to(device)
        res['psnr_codec'].append(psnr_t(x_w, x_c)); res['psnr_orig'].append(psnr_t(x_w, x))
        res['ssim_codec'].append(ssim_torch(x_w, x_c, data_range=1.0).item())
        for a, c in attacks:
            k = f'{a}{"" if c is None else c}'
            if a == 'none':
                xa = x_w
            elif a == 'neural':
                xa = codec_roundtrip(x_w)
            else:
                xa = to_tensor(ffmpeg_roundtrip(x_w_u8, 25, a, c)[0]).to(device)
            logits = extractor(xa)
            acc[k].append(((logits > 0).float() == msgs).float().mean().item())
            vacc[k].append(((logits.mean(0) > 0).float() == msg).float().mean().item())
    out = {k: float(np.mean(v)) for k, v in res.items()}
    out.update({f'acc_{k}': float(np.mean(v)) for k, v in acc.items()})
    out.update({f'vacc_{k}': float(np.mean(v)) for k, v in vacc.items()})
    out['score'] = float(np.mean([np.mean(v) for v in acc.values()]))
    embedder.train(); extractor.train()
    return out


# ------------------------------------------------------------
# UJI KEWARASAN (sanity check): tanpa serangan & tanpa loss citra, pesan acak HARUS bisa
# dipelajari. Jika akurasi tidak naik, ada masalah pada setup (bukan soal robustness).
# ------------------------------------------------------------
def sanity_check(steps=SANITY_STEPS):
    opt_s = torch.optim.Adam(list(embedder.parameters()) + list(extractor.parameters()), lr=LR)
    embedder.train(); extractor.train()
    accs = []
    for i in range(steps):
        x = to_tensor(sample_clips(train_cache)).to(device)
        m = torch.randint(0, 2, (x.size(0), MSG_BITS), device=device).float()
        _log_payloads(m)
        x_w, y0, y_w0 = embed_tensor(x, m)
        logits = extractor(x_w)
        loss_b, _, loss = msg_delta_loss(logits, m, y_w0 - y0)      # [STAGE-1] bit loss + penalti L2 delta
        opt_s.zero_grad(); loss.backward(); opt_s.step()
        accs.append(((logits > 0).float() == m).float().mean().item())
        if (i + 1) % max(1, steps // 6) == 0:
            print(f'  sanity step {i+1}/{steps}: bce={loss_b.item():.3f} bit_acc={np.mean(accs[-20:]):.3f}')
    final = float(np.mean(accs[-20:]))
    if final < 0.85 and STRICT_SANITY:
        raise RuntimeError(f'Sanity check GAGAL (bit_acc={final:.3f} < 0.85). Jalankan Diagnostik Level 2 (6a) dan perbaiki sebelum training penuh.')
    if final < 0.85:
        print(f'[PERINGATAN] bit_acc sanity hanya {final:.3f}. Periksa: bobot codec pretrained termuat? '
              f'FRAME_SIZE terlalu kecil? Coba naikkan SANITY_STEPS atau turunkan LAMBDA_DELTA.')
    else:
        print(f'[OK] Pesan dapat dipelajari (bit_acc {final:.3f}). Lanjut ke training penuh.')
    return final


# [AUDIT-PATCH] catat SEMUA payload yang dipakai training/sanity (bukti payload test unseen)
TRAIN_PAYLOADS = set()
TRAIN_PAYLOADS_PATH = os.path.join(OUTPUT_DIR, 'models', f'train_payloads_{CFG_TAG}.json')
if os.path.exists(TRAIN_PAYLOADS_PATH):
    TRAIN_PAYLOADS = set(json.load(open(TRAIN_PAYLOADS_PATH)))


def _log_payloads(m):
    for r in m.detach().cpu().numpy().astype(np.uint8):
        TRAIN_PAYLOADS.add(int(''.join(map(str, r)), 2))


# ---------------- [STAGE-1] Evaluator A0 / A1 / kontrol (latent langsung; tanpa g_s / g_a) ----------------
ID_SL, CRC_SL = slice(0, 32), slice(32, 48)


def _draw_payloads(kind, n, rng):
    if kind == 'raw':                                    # 48 bit acak murni (distribusi training)
        return rng.randint(0, 2, (n, MSG_BITS)).astype(np.float32)
    return np.stack([build_message(int(rng.randint(0, 2 ** 32, dtype=np.uint64))) for _ in range(n)]).astype(np.float32)


@torch.no_grad()
def collect_latents(emb, cache, kind='raw', n_frames_per_video=64, chunk=16, seed=2026, alpha=ALPHA_INIT):
    '''y, y_w, payload dari video VALIDASI. Tiap frame payload SENDIRI, tanpa voting. g_a hanya dipakai untuk memperoleh y.'''
    rng = np.random.RandomState(seed + (0 if kind == 'raw' else 1))
    out = {'y': [], 'yw': [], 'm': []}
    for name in sorted(cache):
        fr = cache[name][:n_frames_per_video]
        for i in range(0, len(fr), chunk):
            x = to_tensor(fr[i:i + chunk]).to(device)
            m = torch.from_numpy(_draw_payloads(kind, len(x), rng)).to(device)
            y = codec.g_a(x)
            yw, _ = emb(y, m, alpha)
            out['y'].append(y.cpu()); out['yw'].append(yw.cpu()); out['m'].append(m.cpu())
    return {k: torch.cat(v) for k, v in out.items()}


@torch.no_grad()
def read_A0(ext, yw, chunk=16):
    '''A0: y_w -> extractor. TANPA round, g_s, g_a.'''
    return torch.cat([ext.read_latent(yw[i:i + chunk].to(device)).cpu() for i in range(0, len(yw), chunk)])


def read_A1(ext, yw):
    '''A1: round(y_w) -> extractor. Tanpa codec.'''
    return read_A0(ext, torch.round(yw))


def read_CTRL(ext, y):
    '''Kontrol: y asli (tanpa watermark) -> extractor.'''
    return read_A0(ext, y)


def read_CTRL_round(ext, y):
    '''Kontrol: round(y) (tanpa watermark) -> extractor.'''
    return read_A0(ext, torch.round(y))


def a_metrics(logits, m):
    pred = (logits > 0).int().numpy(); gt = m.int().numpy(); ok = pred == gt
    return {'Bit Accuracy': round(float(ok.mean()), 4), 'Exact Match': round(float(ok.all(1).mean()), 4),
            'ID Accuracy': round(float(ok[:, ID_SL].mean()), 4), 'CRC Accuracy': round(float(ok[:, CRC_SL].mean()), 4),
            'N frame': int(len(gt))}


@torch.no_grad()
def evaluate_A0_A1(emb, ext, cache, n_frames_per_video=64, seed=2026):
    emb_tr, ext_tr = emb.training, ext.training
    emb.eval(); ext.eval()
    rows = []
    for kind in ['raw', 'id_crc']:
        D = collect_latents(emb, cache, kind, n_frames_per_video, seed=seed)
        for exp, rnd, lg in [('A0', 'No', read_A0(ext, D['yw'])), ('A1', 'Yes', read_A1(ext, D['yw'])),
                             ('CTRL (y asli, tanpa WM)', 'No', read_CTRL(ext, D['y'])),
                             ('CTRL (round(y), tanpa WM)', 'Yes', read_CTRL_round(ext, D['y']))]:
            rows.append({'Payload': kind, 'Experiment': exp, 'Round': rnd, **a_metrics(lg, D['m'])})
    emb.train(emb_tr); ext.train(ext_tr)
    return pd.DataFrame(rows)


## 5b. [STAGE-1] Verifikasi implementasi (WAJIB sebelum retraining)

Memeriksa secara struktural, runtime, dan behavioral bahwa tiga perubahan benar-benar terjadi. **PASS di sini bukan TRAINING PASS**: akurasi tidak dipakai sebagai bukti implementasi. Model yang diuji dibuat baru dari seed (`SEED`), bukan model yang sudah dilatih. Hasil disimpan ke `stage1_verification.json`.


In [ ]:
# ===== [STAGE-1] VERIFIKASI IMPLEMENTASI: struktural + runtime + behavioral. TANPA training. =====
import re, json, copy, hashlib, inspect, subprocess

VERIF = {}
VALS = {}


def _rec(key, ok, detail=''):
    VERIF[key] = bool(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {key}" + (f' :: {detail}' if detail else ''))


def _src(obj):
    try:
        return inspect.getsource(obj)
    except Exception:
        return None


def _code_only(s):
    s = re.sub(r"'''.*?'''|\"\"\".*?\"\"\"", '', s, flags=re.S)
    return re.sub(r'#.*', '', s)


def _fresh_models():
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    return LatentWatermarkEmbedder().to(device), LatentWatermarkExtractor().to(device)


def _fingerprint(*mods):
    h = hashlib.sha256()
    for m in mods:
        for k, v in m.state_dict().items():
            h.update(k.encode()); h.update(v.detach().cpu().numpy().tobytes())
    return h.hexdigest()[:16]


def _grad_norm(mod):
    sq = [p.grad.detach().pow(2).sum() for p in mod.parameters() if p.grad is not None]
    return float(torch.stack(sq).sum().sqrt()) if sq else 0.0


_saved_models = (embedder, extractor)
emb_v, ext_v = _fresh_models()
embedder, extractor = emb_v, ext_v          # embed_tensor() memakai global `embedder` -> uji jalur kode yang sama
try:
    # ------------------------------------------------------------------ A. Message diversity
    print('== A. Message diversity ==')
    _rec('A1 konfigurasi 16x1', CLIPS_PER_BATCH == 16 and CLIP_LEN == 1, f'CLIPS_PER_BATCH={CLIPS_PER_BATCH} CLIP_LEN={CLIP_LEN}')
    captured = []
    _hk = embedder.register_forward_pre_hook(lambda mod, args: captured.append(args[1].detach().clone().cpu()))
    try:
        msgs_s, msg_f_s = make_step_messages(CLIPS_PER_BATCH, CLIP_LEN)
        x_s = to_tensor(sample_clips(train_cache)).to(device)
        with torch.no_grad():
            _, y_s, yw_s = embed_tensor(x_s, msg_f_s)
    finally:
        _hk.remove()
    cap = captured[0]                                     # payload yang BENAR-BENAR masuk ke embedder
    n_frames, (n_uniq, max_fp) = cap.shape[0], payload_diversity(cap)
    _u200 = [payload_diversity(make_step_messages(CLIPS_PER_BATCH, CLIP_LEN)[1])[0] for _ in range(200)]
    VALS['diversity'] = {'frames_in_embedder': n_frames, 'unique_payloads_per_step': n_uniq, 'max_frames_per_payload': max_fp,
                         'unique_min_over_200_steps': min(_u200)}
    _rec('A2 runtime: 16 frame, 16 unik, 1 frame/payload', n_frames == 16 and n_uniq == 16 and max_fp == 1 and x_s.shape[0] == 16 and min(_u200) == 16,
         f'frames={n_frames} unique={n_uniq} max_frames_per_payload={max_fp} unique_min(200 step)={min(_u200)}')
    print(f'      unique payloads per step = {n_uniq} | frames per payload = {max_fp}')

    _pid = make_watermark_id(OWNER_TEXT, 'verif.mp4', '2026-01-01T00:00:00')
    _bm = build_message(_pid); _crc = crc16_ccitt(_pid.to_bytes(ID_BITS // 8, 'big'))
    _dec = decode_message((_bm * 2 - 1) * 5)
    _ok_a3 = (MSG_BITS == 48 and ID_BITS == 32 and CRC_BITS == 16 and ECC_REPEAT == 1 and _bm.shape == (48,)
              and bits_to_int(_bm[:32].astype(int)) == _pid and bits_to_int(_bm[32:].astype(int)) == _crc
              and _dec['crc_ok'] and _dec['id'] == _pid and msg_f_s.shape[1] == 48)
    _rec('A3 payload 48 bit = 32 ID + 16 CRC', _ok_a3, f'MSG_BITS={MSG_BITS} ID={ID_BITS} CRC={CRC_BITS}')

    # ------------------------------------------------------------------ B. Embedder
    print('== B. Embedder ==')
    _bad_mod = [type(m).__name__ for m in emb_v.modules() if isinstance(m, (nn.Tanh, nn.Sigmoid, nn.Hardtanh, nn.ReLU6, nn.Hardsigmoid, nn.Softsign))]
    _fw = _src(LatentWatermarkEmbedder.forward)
    _tok = sorted(set(re.findall(r'tanh|sigmoid|clamp|clip|DELTA_MAX', _code_only(_fw)))) if _fw else None
    _cap_out = []
    _h2 = emb_v.final_layer().register_forward_hook(lambda m, i, o: _cap_out.append(o.detach()))
    with torch.no_grad():
        yw_b, dlt_b = emb_v(y_s, msg_f_s)
    _h2.remove()
    _raw_out = torch.equal(_cap_out[0], dlt_b)             # delta == output mentah layer terakhir
    emb_big = copy.deepcopy(emb_v)                          # uji tanpa saturasi: bobot akhir x1e4 -> |delta| harus melewati DELTA_MAX
    with torch.no_grad():
        emb_big.final_layer().weight.mul_(1e4)
        _, d_big = emb_big(y_s, msg_f_s)
    _nosat = float(d_big.abs().max()) > 1.5 * DELTA_MAX
    _rec('B1 delta mentah (tanpa tanh/sigmoid/clamp)', (not _bad_mod) and (_tok in (None, [])) and _raw_out and _nosat,
         f'modul_terlarang={_bad_mod} token_forward={_tok} delta==output_layer_akhir={_raw_out} max|delta| saat bobot x1e4={float(d_big.abs().max()):.1f} (> {1.5*DELTA_MAX})')

    yb, mb = y_s.detach(), msg_f_s.detach()
    yw2, dlt2 = emb_v(yb, mb)
    lg2 = ext_v.read_latent(ste_round(yw2))
    bit_l, d_l, tot_l = msg_delta_loss(lg2, mb, dlt2)      # fungsi yang sama dipakai sanity/diagnostik/training penuh
    W = emb_v.final_layer().weight
    g_d, = torch.autograd.grad(d_l, W, retain_graph=True, allow_unused=True)
    g_b, = torch.autograd.grad(bit_l, W, retain_graph=True, allow_unused=True)
    g_t, = torch.autograd.grad(tot_l, W, retain_graph=True, allow_unused=True)
    g_all = torch.autograd.grad(d_l, list(emb_v.parameters()), retain_graph=True, allow_unused=True)
    n_par_grad = sum(1 for g in g_all if g is not None and float(g.abs().sum()) > 0)
    _dl_ok = (float(d_l.detach()) > 0 and d_l.requires_grad and g_d is not None and float(g_d.norm()) > 0
              and abs(float(tot_l) - (float(bit_l) + LAMBDA_DELTA * float(d_l))) < 1e-6
              and torch.allclose(g_t, g_b + LAMBDA_DELTA * g_d, rtol=1e-4, atol=1e-10) and n_par_grad >= 1)
    VALS['delta_l2'] = {'delta_loss': float(d_l), 'grad_norm_final_W_from_delta_loss': float(g_d.norm()) if g_d is not None else None,
                        'params_with_grad_from_delta_loss': n_par_grad, 'LAMBDA_DELTA': LAMBDA_DELTA}
    _rec('B2 delta L2 masuk total_loss + gradient', _dl_ok,
         f'delta_loss={float(d_l):.3e} >0 | |grad_W(delta_loss)|={float(g_d.norm()):.3e} | param embedder ber-gradient dari delta_loss={n_par_grad} | total=bit+{LAMBDA_DELTA}*delta')

    Lf = emb_v.final_layer(); w, b = Lf.weight.detach(), Lf.bias.detach()
    INIT = {'weight_mean': float(w.mean()), 'weight_std': float(w.std()), 'bias_mean': float(b.mean()),
            'bias_std': float(b.std()), 'bias_absmax': float(b.abs().max())}
    VALS['init_final_layer'] = INIT
    _rec('B3 init layer akhir (std~1e-3, bias 0)', 0.9e-3 <= INIT['weight_std'] <= 1.1e-3 and abs(INIT['weight_mean']) < 1e-4 and INIT['bias_absmax'] == 0.0
         and Lf.out_channels == LATENT_CH, ', '.join(f'{k}={v:.3e}' for k, v in INIT.items()))

    _fin = bool(torch.isfinite(dlt_b).all()); _nan = bool(torch.isnan(dlt_b).any()); _inf = bool(torch.isinf(dlt_b).any())
    _rec('B4 delta finite & shape == latent', _fin and not _nan and not _inf and dlt_b.shape == y_s.shape and dlt_b.dtype == y_s.dtype and dlt_b.device == y_s.device,
         f'finite={_fin} nan={_nan} inf={_inf} shape(delta)={tuple(dlt_b.shape)} shape(latent)={tuple(y_s.shape)} mean|delta|={float(dlt_b.abs().mean()):.3e}')

    # ------------------------------------------------------------------ C. Extractor
    print('== C. Extractor ==')
    _mods = list(ext_v.modules())
    _old = [type(m).__name__ for m in _mods if isinstance(m, (nn.AdaptiveAvgPool2d, nn.AdaptiveMaxPool2d, nn.Flatten, nn.Linear))]
    _sig = [type(m).__name__ for m in _mods if isinstance(m, (nn.Sigmoid, nn.Softmax, nn.LogSoftmax))]
    _convs = [m for m in _mods if isinstance(m, nn.Conv2d)]
    _fc = []
    _h3 = _convs[-1].register_forward_hook(lambda m, i, o: _fc.append(o.detach()))
    with torch.no_grad():
        lg_c = ext_v.read_latent(ste_round(yw_b))
    _h3.remove()
    feats = _fc[0]
    _rec('C1 sebelum pooling [B, 48, H, W]', feats.ndim == 4 and feats.shape[1] == MSG_BITS == 48 and _convs[-1].out_channels == 48, f'feats.shape={tuple(feats.shape)}')
    _gap = torch.allclose(lg_c, feats.mean(dim=(-2, -1)), atol=1e-6)
    _rec('C2 GAP -> [B, 48]', _gap and tuple(lg_c.shape) == (feats.shape[0], 48), f'logits.shape={tuple(lg_c.shape)} logits==feats.mean(dim=(-2,-1)) : {_gap}')
    _rec('C3 tanpa AdaptiveAvgPool/Flatten/Linear', not _old, f'modul_lama_ditemukan={_old}')
    _fw_ext = _src(LatentWatermarkExtractor)
    _sig_src = re.findall(r'sigmoid|softmax', _code_only(_fw_ext).lower()) if _fw_ext else None
    _fin_c = bool(torch.isfinite(lg_c).all())
    _rec('C4 raw logits [B,48], finite, tanpa sigmoid', not _sig and _sig_src in (None, []) and _fin_c and tuple(lg_c.shape)[1] == 48,
         f'shape={tuple(lg_c.shape)} finite={_fin_c} min={float(lg_c.min()):.3f} max={float(lg_c.max()):.3f} modul_sigmoid={_sig}')
    with torch.no_grad():
        _lg_pix = ext_v(x_s)
    VALS['extractor_forward_pixel_shape'] = list(_lg_pix.shape)

    # ------------------------------------------------------------------ D/F. Smoke forward-backward + leakage
    print('== D/F. Forward/backward smoke test (1 batch) + leakage ==')
    emb_v, ext_v = _fresh_models(); embedder, extractor = emb_v, ext_v
    cap2 = []
    _hk = emb_v.register_forward_pre_hook(lambda mod, args: cap2.append(args[1].detach().clone()))
    msgs_d, msg_f_d = make_step_messages(CLIPS_PER_BATCH, CLIP_LEN)
    msg_before = msg_f_d.clone()
    x_d = to_tensor(sample_clips(train_cache)).to(device)
    with torch.no_grad():
        y_d = codec.g_a(x_d)
    yw_d, dlt_d = emb_v(y_d, msg_f_d, ALPHA_INIT)
    _hk.remove()
    lg_d = ext_v.read_latent(ste_round(yw_d))
    bit_d, dl_d, tot_d = msg_delta_loss(lg_d, msg_f_d, dlt_d)
    tot_d.backward()
    SMOKE = {'total_loss': float(tot_d), 'bit_loss': float(bit_d), 'delta_loss': float(dl_d),
             'mean_abs_delta': float(dlt_d.abs().mean()), 'max_abs_delta': float(dlt_d.abs().max()),
             'embedder_grad_norm': _grad_norm(emb_v), 'extractor_grad_norm': _grad_norm(ext_v)}
    VALS['smoke'] = SMOKE
    for k, v in SMOKE.items():
        print(f'      {k:<22s} {v:.6e}')
    _nan_grad = any(torch.isnan(p.grad).any() or torch.isinf(p.grad).any() for m in (emb_v, ext_v) for p in m.parameters() if p.grad is not None)
    _rec('D forward: loss finite, shape valid', all(np.isfinite(list(SMOKE.values()))) and lg_d.shape == (16, 48) and yw_d.shape == y_d.shape,
         f'logits={tuple(lg_d.shape)} y_w={tuple(yw_d.shape)}')
    _rec('D backward: grad embedder & extractor != 0, tanpa NaN/Inf', SMOKE['embedder_grad_norm'] > 0 and SMOKE['extractor_grad_norm'] > 0 and not _nan_grad,
         f'emb_grad={SMOKE["embedder_grad_norm"]:.3e} ext_grad={SMOKE["extractor_grad_norm"]:.3e} nan_or_inf_grad={_nan_grad}')

    gt = msg_f_d.detach().cpu().int(); pred = (lg_d.detach().cpu() > 0).int()
    print('      payload_ground_truth[:16] / predicted_bits[:16] (48 bit; model masih inisialisasi, akurasi ~0.5 wajar):')
    for i in range(16):
        print(f'        GT  {"".join(map(str, gt[i].tolist()))}\n        PRD {"".join(map(str, pred[i].tolist()))}')
    print('      logits[:16, :8] =\n', np.round(lg_d.detach().cpu().numpy()[:16, :8], 3))
    _leak_ok = torch.equal(cap2[0], msg_before) and torch.equal(msg_f_d, msg_before)
    _leak_ok = _leak_ok and abs(F.binary_cross_entropy_with_logits(lg_d.detach(), msg_before).item() - bit_d.item()) < 1e-6
    _rec('F leakage: target loss == payload yang di-embed, tidak dimodifikasi', _leak_ok, 'payload masuk embedder == target loss == payload asli (tidak di-mutasi)')

    # ------------------------------------------------------------------ E. A0 / A1 / kontrol
    print('== E. Pipeline A0 / A1 / kontrol ==')
    Dv = collect_latents(emb_v, val_cache, 'raw', n_frames_per_video=8)
    calls = {'g_s': 0, 'g_a': 0, 'embedder': 0}
    _hs = [codec.g_s.register_forward_hook(lambda *a: calls.__setitem__('g_s', calls['g_s'] + 1)),
           codec.g_a.register_forward_hook(lambda *a: calls.__setitem__('g_a', calls['g_a'] + 1)),
           emb_v.register_forward_hook(lambda *a: calls.__setitem__('embedder', calls['embedder'] + 1))]
    seen = {}
    _pre = ext_v.net.register_forward_pre_hook(lambda mod, args: seen.__setitem__('x', (args[0] * LATENT_SCALE).detach().cpu()))
    lgA0 = read_A0(ext_v, Dv['yw']); xA0 = seen['x']
    _isint = lambda t: bool(torch.allclose(t, torch.round(t), atol=1e-3))
    lgA1 = read_A1(ext_v, Dv['yw']); xA1 = seen['x']
    lgC0 = read_CTRL(ext_v, Dv['y']); xC0 = seen['x']
    lgC1 = read_CTRL_round(ext_v, Dv['y']); xC1 = seen['x']
    _pre.remove(); [h.remove() for h in _hs]
    _nA1, _nC0 = xA1.shape[0], xC0.shape[0]   # hook hanya menyimpan chunk TERAKHIR (bisa < 16, mis. 8) -> bandingkan dg ekor sepanjang itu
    _ref = lambda y: torch.cat([ext_v.read_latent(y[i:i + 16].to(device)).detach().cpu() for i in range(0, len(y), 16)])
    _rec('E1 A0: y_w -> extractor (tanpa round/g_s/g_a)', calls['g_s'] == 0 and calls['g_a'] == 0 and not _isint(xA0) and torch.allclose(lgA0, _ref(Dv['yw']), atol=1e-5),
         f'panggilan g_s={calls["g_s"]} g_a={calls["g_a"]} | input extractor non-integer={not _isint(xA0)}')
    _rec('E2 A1: round(y_w) -> extractor (tanpa codec)', calls['g_s'] == 0 and calls['g_a'] == 0 and _isint(xA1) and xA1.shape[0] == _nA1 and torch.allclose(xA1, torch.round(Dv['yw'])[-_nA1:], atol=1e-3),
         f'input extractor integer-valued={_isint(xA1)}')
    _rec('E3 kontrol y & round(y) tanpa watermark tersedia', calls['embedder'] == 0 and xC0.shape[0] == _nC0 and torch.allclose(xC0, Dv['y'][-_nC0:], atol=1e-3) and _isint(xC1),
         f'panggilan embedder saat kontrol={calls["embedder"]} | evaluator tunggal read_A0/A1/CTRL dipakai bersama diagnostik')

    # ------------------------------------------------------------------ G. Reproducibility
    print('== G. Reproducibility ==')
    fpA = _fingerprint(*_fresh_models()); fpB = _fingerprint(*_fresh_models())
    try:
        _git = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, timeout=5).stdout.strip() or 'n/a (bukan repo git)'
    except Exception:
        _git = 'n/a (git tidak tersedia)'
    REPRO = {'seed': SEED, 'notebook_version': NOTEBOOK_VERSION, 'git_commit': _git, 'CFG_TAG': CFG_TAG,
             'model_init_fingerprint': fpA, 'CLIPS_PER_BATCH': CLIPS_PER_BATCH, 'CLIP_LEN': CLIP_LEN, 'MSG_BITS': MSG_BITS,
             'BPDA_CLIPS_PER_BATCH': BPDA_CLIPS_PER_BATCH, 'BPDA_CLIP_LEN': BPDA_CLIP_LEN, 'LAMBDA_DELTA': LAMBDA_DELTA,
             'LR': LR, 'STAGE1_DIAG_STEPS': STAGE1_DIAG_STEPS}
    VALS['reproducibility'] = REPRO
    _rec('G seed & konfigurasi tercatat, init deterministik', fpA == fpB and SEED is not None, f'seed={SEED} init_fingerprint={fpA} (ulang: {fpB}) commit={_git}')
finally:
    embedder, extractor = _saved_models

_NAMES = [('Message diversity       ', ['A1 konfigurasi 16x1', 'A2 runtime: 16 frame, 16 unik, 1 frame/payload']),
          ('Payload integrity       ', ['A3 payload 48 bit = 32 ID + 16 CRC']),
          ('Embedder raw delta      ', ['B1 delta mentah (tanpa tanh/sigmoid/clamp)', 'B4 delta finite & shape == latent']),
          ('Delta L2 loss           ', ['B2 delta L2 masuk total_loss + gradient']),
          ('Final-layer init        ', ['B3 init layer akhir (std~1e-3, bias 0)']),
          ('Extractor architecture  ', ['C1 sebelum pooling [B, 48, H, W]', 'C2 GAP -> [B, 48]', 'C3 tanpa AdaptiveAvgPool/Flatten/Linear']),
          ('Logits shape            ', ['C4 raw logits [B,48], finite, tanpa sigmoid']),
          ('Forward smoke test      ', ['D forward: loss finite, shape valid']),
          ('Backward smoke test     ', ['D backward: grad embedder & extractor != 0, tanpa NaN/Inf']),
          ('A0 pipeline preserved   ', ['E1 A0: y_w -> extractor (tanpa round/g_s/g_a)']),
          ('A1 pipeline preserved   ', ['E2 A1: round(y_w) -> extractor (tanpa codec)']),
          ('Control preserved       ', ['E3 kontrol y & round(y) tanpa watermark tersedia']),
          ('Data leakage check      ', ['F leakage: target loss == payload yang di-embed, tidak dimodifikasi']),
          ('Reproducibility         ', ['G seed & konfigurasi tercatat, init deterministik'])]
print('\nIMPLEMENTATION VERIFICATION\n============================\n')
_failed = []
for label, keys in _NAMES:
    ok = all(VERIF.get(k, False) for k in keys)
    print(f'{label}: {"PASS" if ok else "FAIL"}')
    if not ok:
        _failed += [k for k in keys if not VERIF.get(k, False)]
STAGE1_IMPL_PASS = not _failed
print(f'\nOVERALL IMPLEMENTATION:\n{"PASS" if STAGE1_IMPL_PASS else "FAIL"}\n\nReason:\n' + ('-' if STAGE1_IMPL_PASS else '\n'.join(' - ' + k for k in _failed)))
print('\n(IMPLEMENTATION PASS != TRAINING PASS. Training harus membuktikan A0 > 0.98 dan A1 > 0.98, lalu Uji B > 0.95.)')
with open(os.path.join(OUTPUT_DIR, 'stage1_verification.json'), 'w') as _f:
    json.dump({'checks': VERIF, 'overall_pass': STAGE1_IMPL_PASS, 'values': VALS}, _f, indent=1, default=str)


## 5c. [STAGE-1] Training diagnostik + A0/A1 (retraining Tahap 1)

Hanya jalan bila sel verifikasi di atas `OVERALL IMPLEMENTATION: PASS`. Setup: **tanpa** ffmpeg/HEVC/H.264/AV1, **tanpa** serangan, **tanpa** loss citra (MSE/SSIM/rate), `STAGE1_DIAG_STEPS` step, **16 payload unik / step, 1 frame / payload** (di-`assert` tiap step).

Loss = `BCE(extractor(round_STE(y_w)))` + `BCE(extractor(y_w))` + `LAMBDA_DELTA * delta.pow(2).mean()`. Dua cabang bit-loss dipakai agar extractor yang sama dilatih untuk bacaan **A1** (round) dan **A0** (tanpa round); keputusan ini dicatat di hasil. Evaluasi: video **validasi**, payload segar, satu payload per frame, tanpa voting; A0/A1 + kontrol dengan payload `raw` (48 bit acak) dan `id_crc` (format ID+CRC-16). Ambang customer **tidak diubah** (A0 > 0.98, A1 > 0.98); Uji B **belum** dijalankan di sel ini.


In [ ]:
# ===== [STAGE-1] TRAINING DIAGNOSTIK (tanpa ffmpeg/codec video/attack/image-loss) + evaluasi A0/A1 =====
import json, time
assert 'STAGE1_IMPL_PASS' in globals() and STAGE1_IMPL_PASS, 'Verifikasi implementasi belum PASS -> perbaiki dulu; retraining dilarang.'

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
s1_emb, s1_ext = LatentWatermarkEmbedder().to(device), LatentWatermarkExtractor().to(device)
s1_opt = torch.optim.Adam(list(s1_emb.parameters()) + list(s1_ext.parameters()), lr=LR)
S1_LOG, s1_acc0, s1_acc1 = [], [], []
S1_DIV = {'uniq_min': 10 ** 9, 'maxfp': 0, 'steps': 0}
s1_emb.train(); s1_ext.train()
_t0 = time.time()
for step in range(1, STAGE1_DIAG_STEPS + 1):
    x = to_tensor(sample_clips(train_cache, CLIPS_PER_BATCH, CLIP_LEN)).to(device)          # 16 frame
    msgs, msg_f = make_step_messages(CLIPS_PER_BATCH, CLIP_LEN)                              # 16 payload independen & unik
    _log_payloads(msgs)
    _u, _fp = payload_diversity(msg_f)                                                       # payload aktual yang masuk model
    assert _u == CLIPS_PER_BATCH == 16 and _fp == 1 and msg_f.shape == (16, MSG_BITS), (_u, _fp)
    S1_DIV['uniq_min'] = min(S1_DIV['uniq_min'], _u); S1_DIV['maxfp'] = max(S1_DIV['maxfp'], _fp); S1_DIV['steps'] += 1
    with torch.no_grad():
        y = codec.g_a(x)
    y_w, delta = s1_emb(y, msg_f, ALPHA_INIT)
    lg1 = s1_ext.read_latent(ste_round(y_w))                                                 # cabang A1 (round, STE)
    lg0 = s1_ext.read_latent(y_w)                                                            # cabang A0 (tanpa round)
    bit1, dl, tot1 = msg_delta_loss(lg1, msg_f, delta)
    bit0 = F.binary_cross_entropy_with_logits(lg0, msg_f)
    loss = tot1 + bit0
    s1_opt.zero_grad(); loss.backward(); s1_opt.step()
    s1_acc1.append(((lg1 > 0).float() == msg_f).float().mean().item()); s1_acc0.append(((lg0 > 0).float() == msg_f).float().mean().item())
    if step % 100 == 0 or step == 1:
        rec = {'step': step, 'loss': loss.item(), 'bit_A1': bit1.item(), 'bit_A0': bit0.item(), 'delta_l2': dl.item(),
               'mean_abs_delta': delta.abs().mean().item(), 'max_abs_delta': delta.abs().max().item(),
               'train_acc_A1': float(np.mean(s1_acc1[-50:])), 'train_acc_A0': float(np.mean(s1_acc0[-50:])),
               'unique_payloads_per_step': _u, 'frames_per_payload': _fp}
        S1_LOG.append(rec)
        print(f'step {step:5d} | loss={rec["loss"]:.4f} | train_acc A0={rec["train_acc_A0"]:.3f} A1={rec["train_acc_A1"]:.3f} | '
              f'|delta| mean={rec["mean_abs_delta"]:.3f} max={rec["max_abs_delta"]:.2f} | uniq/step={_u} frames/payload={_fp}')
print(f'\nTraining selesai: {STAGE1_DIAG_STEPS} step, {time.time()-_t0:.0f}s. Bukti diversity (semua step): '
      f'unique payloads per step (min) = {S1_DIV["uniq_min"]} | frames per payload (max) = {S1_DIV["maxfp"]} | steps = {S1_DIV["steps"]}')

S1_CKPT = os.path.join(OUTPUT_DIR, 'models', f'stage1_diag_{CFG_TAG}.pth')
S1_CFG = {'seed': SEED, 'notebook_version': NOTEBOOK_VERSION, 'CFG_TAG': CFG_TAG, 'steps': STAGE1_DIAG_STEPS, 'CLIPS_PER_BATCH': CLIPS_PER_BATCH,
          'CLIP_LEN': CLIP_LEN, 'MSG_BITS': MSG_BITS, 'LAMBDA_DELTA': LAMBDA_DELTA, 'LR': LR, 'ffmpeg': False, 'attack': False, 'image_loss': False,
          'loss': 'BCE(A1: round STE) + BCE(A0: tanpa round) + LAMBDA_DELTA*delta.pow(2).mean()'}
torch.save({'embedder': s1_emb.state_dict(), 'extractor': s1_ext.state_dict(), 'cfg': S1_CFG}, S1_CKPT)

# ---- Evaluasi A0/A1 + kontrol (video validasi, payload segar, 1 payload/frame, tanpa voting)
T_S1 = evaluate_A0_A1(s1_emb, s1_ext, val_cache)
print('\n=== A0 / A1 / KONTROL (checkpoint diagnostik Tahap 1; video VALIDASI) ===')
print(T_S1.to_string(index=False))
_g = lambda kind, exp: float(T_S1[(T_S1['Payload'] == kind) & (T_S1['Experiment'] == exp)]['Bit Accuracy'].iloc[0])
S1_RES = {'A0': {k: _g(k, 'A0') for k in ['raw', 'id_crc']}, 'A1': {k: _g(k, 'A1') for k in ['raw', 'id_crc']}}
_a0_ok = all(v > TARGET_A for v in S1_RES['A0'].values()) if 'TARGET_A' in globals() else all(v > 0.98 for v in S1_RES['A0'].values())
_a1_ok = all(v > TARGET_A for v in S1_RES['A1'].values()) if 'TARGET_A' in globals() else all(v > 0.98 for v in S1_RES['A1'].values())
print(f'\nA0 bit acc {S1_RES["A0"]} -> target > 0.98 : {"tercapai" if _a0_ok else "BELUM tercapai"}')
print(f'A1 bit acc {S1_RES["A1"]} -> target > 0.98 : {"tercapai" if _a1_ok else "BELUM tercapai"}')
if not _a0_ok:
    print('-> A0 rendah: masalah masih di embedder / extractor / loss / sinyal training / message diversity. JANGAN lanjut ke codec/HEVC.')
elif not _a1_ok:
    print('-> A0 tinggi tetapi A1 turun: investigasi efek quantization/rounding (delta vs langkah kuantisasi). Belum ke Uji B.')
else:
    print('-> A0 dan A1 memenuhi ambang pada validasi. Baru boleh lanjut ke Uji B (round -> g_s -> g_a -> extractor; target > 0.95). Ini BUKAN klaim robustness/generalisasi.')
with open(os.path.join(OUTPUT_DIR, 'stage1_diag_results.json'), 'w') as _f:
    json.dump({'config': S1_CFG, 'diversity': S1_DIV, 'train_log': S1_LOG, 'A0_A1_table': T_S1.to_dict('records'), 'summary': S1_RES,
               'gate_A0': _a0_ok, 'gate_A1': _a1_ok}, _f, indent=1)


## 6a. Diagnostik Level 2 (jalankan SEBELUM training penuh)

Melatih embedder+extractor **baru** (tidak menyentuh model utama) tanpa serangan dan tanpa loss citra, dengan tiga titik baca:

| Mode | Jalur | Jika gagal berarti |
|---|---|---|
| `latent_float` | `y_w` dibaca langsung (tanpa g_s/g_a) | embedder/extractor belum bisa belajar (optimisasi, kapasitas, injeksi pesan) |
| `video_no_round` | `g_s(y_w)` -> `g_a` -> extractor | informasi hilang di siklus g_s->g_a |
| `video_round_ste` | `g_s(STE-round(y_w))` -> `g_a` -> extractor | kuantisasi menghapus perturbasi (jalur yang dipakai training sebenarnya) |

In [ ]:
DIAG_MODES = ['latent_float', 'video_no_round', 'video_round_ste']


class LatentDirectExtractor(LatentWatermarkExtractor):
    """Sama dengan extractor asli, tetapi membaca latent langsung (tanpa g_a)."""
    def forward(self, y):
        return self.read_latent(y)


def run_diag(mode, steps=DIAG_STEPS, alpha=ALPHA_INIT):
    torch.manual_seed(SEED)
    emb = LatentWatermarkEmbedder().to(device)
    ext = (LatentDirectExtractor() if mode == 'latent_float' else LatentWatermarkExtractor()).to(device)
    opt = torch.optim.Adam(list(emb.parameters()) + list(ext.parameters()), lr=LR)
    bce_ = nn.BCEWithLogitsLoss()
    accs = []
    for i in range(steps):
        x = to_tensor(sample_clips(train_cache)).to(device)
        m = torch.randint(0, 2, (x.size(0), MSG_BITS), device=device).float()
        with torch.no_grad():
            y = codec.g_a(x)
        y_w, dlt = emb(y, m, alpha)
        if mode == 'latent_float':
            lg = ext(y_w)
        elif mode == 'video_no_round':
            lg = ext(codec.g_s(y_w).clamp(0, 1))
        else:
            lg = ext(codec.g_s(ste_round(y_w)).clamp(0, 1))
        loss = bce_(lg, m) + LAMBDA_DELTA * delta_l2(dlt)   # [STAGE-1]
        opt.zero_grad(); loss.backward(); opt.step()
        accs.append(((lg > 0).float() == m).float().mean().item())
        if (i + 1) % max(1, steps // 4) == 0:
            print(f'  [{mode}] step {i+1}/{steps} bit_acc(50 terakhir)={np.mean(accs[-50:]):.3f}')
    return float(np.mean(accs[-50:]))


DIAG_RESULT = {m: run_diag(m) for m in DIAG_MODES}
print('\nHASIL DIAGNOSTIK LEVEL 2 (bit_acc tanpa serangan, tanpa loss citra):', {k: round(v, 3) for k, v in DIAG_RESULT.items()})
if DIAG_RESULT['latent_float'] < 0.85:
    print('-> Embedder/extractor belum bisa belajar bahkan TANPA g_s/g_a. Fokus: optimisasi (LR, langkah), kapasitas, cara injeksi pesan, LAMBDA_DELTA.')
elif DIAG_RESULT['video_no_round'] < 0.85:
    print('-> Latent bisa dibaca, tetapi informasi hilang di siklus g_s->g_a. Fokus: jalur pixel, kekuatan delta, loss konsistensi latent.')
elif DIAG_RESULT['video_round_ste'] < 0.85:
    print('-> Kuantisasi menghapus perturbasi. Fokus: skala delta (LAMBDA_DELTA) / margin terhadap langkah kuantisasi.')
else:
    print('-> Jalur penuh dapat belajar dalam DIAG_STEPS. Lanjut ke training penuh.')
del DIAG_MODES

## 6b. [AUDIT v2 — Aksi 1] Uji A / B / C dengan 16 pesan unik per step

Jalankan **setelah Diagnostik Level 2 (6a)** dan **sebelum training penuh (6)**. Melatih model diagnostik BARU (tidak menyentuh `embedder`/`extractor` utama), hanya memakai `train_cache` dan `val_cache` (bukan video test).

| Label | Jalur | Konfigurasi |
|---|---|---|
| Uji A | `round(y_w)` → extractor (tanpa `g_s`/`g_a`) | 16×1 |
| Uji B | `round(y_w)` → `g_s` → uint8 → `g_a` → extractor | 16×1 |
| Uji A (2×8) | jalur Uji A | 2×8 |
| Uji C | jalur Uji B | 2×8 |

Threshold customer tidak diubah (A >0,98; B >0,95). PASS pada model diagnostik belum berarti checkpoint utama PASS.

In [ ]:
import random, time
import torch, torch.nn as nn

TARGET_A, TARGET_B = 0.98, 0.95     # threshold customer, TIDAK diubah (didefinisikan di sini karena sel ini jalan lebih dulu dari 7d)

# (clips_per_batch, clip_len): jumlah frame per step sama (=16) -> B dan C comparable
CFG_B = (16, 1)     # Uji B: 16 pesan berbeda / step, tiap frame pesannya sendiri
CFG_C = (2, 8)      # Uji C: konfigurasi lama
DIAG2_STEPS = 600   # jumlah step training diagnostik (model BARU, bukan model utama)
DIAG2_EVAL_BATCHES = 60   # 60 x 16 = 960 frame evaluasi, pesan segar tiap batch


def unique_messages(n_msg, bits=None):
    bits = bits or MSG_BITS
    for _ in range(100):
        m = torch.randint(0, 2, (n_msg, bits), device=device)
        if len({tuple(r) for r in m.int().tolist()}) == n_msg:
            return m.float()
    raise RuntimeError('gagal membuat pesan unik')


def _to_u8_ste(x):
    """Pembulatan ke uint8 (seperti jalur file/evaluasi) dengan STE."""
    return x + (torch.round(x * 255) / 255 - x).detach()


def _readout(mode, emb, ext, y, m, alpha, hard, return_delta=False):
    y_w, delta = emb(y, m, alpha)
    yq = torch.round(y_w) if hard else ste_round(y_w)
    if mode == 'latent_round_ste':                      # UJI A: round(y_w) -> extractor (tanpa g_s/g_a)
        return (ext(yq), delta) if return_delta else ext(yq)
    xr = codec.g_s(yq).clamp(0, 1)                      # UJI B/C: round(y_w) -> g_s -> (uint8) -> g_a -> extractor
    xr = to_tensor(to_uint8(xr)).to(device) if hard else _to_u8_ste(xr)
    return (ext(xr), delta) if return_delta else ext(xr)


def run_diag_v2(mode, cfg, steps=DIAG2_STEPS, alpha=ALPHA_INIT):
    n_clips, clip_len = cfg
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    emb = LatentWatermarkEmbedder().to(device)
    ext = (LatentDirectExtractor() if mode == 'latent_round_ste' else LatentWatermarkExtractor()).to(device)
    opt = torch.optim.Adam(list(emb.parameters()) + list(ext.parameters()), lr=LR)
    bce_ = nn.BCEWithLogitsLoss()
    accs, uniq_log = [], []
    emb.train(); ext.train()
    for i in range(steps):
        x = to_tensor(sample_clips(train_cache, n_clips, clip_len)).to(device)
        m = unique_messages(n_clips)                                  # n_clips pesan BERBEDA
        uniq_log.append(len({tuple(r) for r in m.int().tolist()}))
        m_f = m.repeat_interleave(clip_len, 0)                        # pesan -> frame
        with torch.no_grad():
            y = codec.g_a(x)
        lg, dl_ = _readout(mode, emb, ext, y, m_f, alpha, hard=False, return_delta=True)
        loss = bce_(lg, m_f) + LAMBDA_DELTA * delta_l2(dl_)   # [STAGE-1]
        opt.zero_grad(); loss.backward(); opt.step()
        accs.append(((lg > 0).float() == m_f).float().mean().item())
        if (i + 1) % max(1, steps // 4) == 0:
            print(f'  [{mode} {n_clips}x{clip_len}] step {i+1}/{steps} train_bit_acc={np.mean(accs[-50:]):.3f}')
    # ---- evaluasi: model beku, pesan segar (tak pernah dipakai training), video VALIDASI ----
    emb.eval(); ext.eval()
    P, G = [], []
    with torch.no_grad():
        for _ in range(DIAG2_EVAL_BATCHES):
            x = to_tensor(sample_clips(val_cache, 16, 1)).to(device)   # 16 frame, 16 pesan unik, per-frame (tanpa voting)
            m = unique_messages(16)
            lg = _readout(mode, emb, ext, codec.g_a(x), m, alpha, hard=True)
            P.append((lg > 0).int().cpu().numpy()); G.append(m.int().cpu().numpy())
    P, G = np.concatenate(P), np.concatenate(G)
    per_bit = (P == G).mean(0)
    return {'mode': mode, 'cfg': f'{n_clips}x{clip_len}', 'msgs_per_step': int(np.min(uniq_log)),
            'train_bit_acc_last50': float(np.mean(accs[-50:])), 'eval_bit_acc': float(per_bit.mean()),
            'eval_ber': float(1 - per_bit.mean()), 'eval_exact_match': float(np.mean((P == G).all(1))),
            'eval_acc_per_bit': per_bit.round(3).tolist(), 'n_eval_frames': int(len(P))}


DIAG2 = []
for _mode, _cfg, _uji in [('latent_round_ste', CFG_B, 'Uji A'), ('video_round_ste', CFG_B, 'Uji B'),
                          ('latent_round_ste', CFG_C, 'Uji A (2x8)'), ('video_round_ste', CFG_C, 'Uji C')]:
    _t = time.time(); _r = run_diag_v2(_mode, _cfg); _r['uji'] = _uji; DIAG2.append(_r)
    print(f'>> {_uji}: {_r["cfg"]} msgs/step={_r["msgs_per_step"]} eval_bit_acc={_r["eval_bit_acc"]:.4f} '
          f'exact={_r["eval_exact_match"]:.3f} ({time.time()-_t:.0f}s)')
with open(os.path.join(OUTPUT_DIR, 'audit_v2_results.json'), 'w') as _f:
    json.dump(DIAG2, _f, indent=1)

print('\nRINGKAS (bandingkan dengan target customer; bukan klaim PASS bila DIAG_STEPS kecil):')
for r in DIAG2:
    thr = TARGET_A if r['mode'] == 'latent_round_ste' else TARGET_B
    print(f'{r["uji"]:<12} {r["cfg"]:<5} msgs/step={r["msgs_per_step"]:<3} bit_acc={r["eval_bit_acc"]:.4f} '
          f'target>{thr} -> {"PASS" if r["eval_bit_acc"] > thr else "FAIL"}')

## 6c. [AUDIT v2 — Aksi 2] Arsip checkpoint lama + panduan checkpoint bertag konfigurasi

Set `ARCHIVE_OLD_RUN = True` **sekali** agar checkpoint lama (2×8) dipindah dan sanity check + `_log_payloads` benar-benar berjalan di sel training. Sel training (bagian 6) **belum diubah**: edit manual sesuai komentar di akhir sel ini (`CFG_TAG`, path checkpoint, `cfg` di dalam checkpoint, assert pesan unik).

In [ ]:
import shutil

ARCHIVE_OLD_RUN = False   # set True SEKALI untuk memindahkan checkpoint lama (2x8) agar sanity & log payload berjalan

MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
if ARCHIVE_OLD_RUN:
    _arch = os.path.join(MODELS_DIR, 'old_run_2x8'); os.makedirs(_arch, exist_ok=True)
    for _fn in ['wm_checkpoint.pth', 'wm_best.pth', 'train_payloads.json']:
        _p = os.path.join(MODELS_DIR, _fn)
        if os.path.exists(_p):
            shutil.move(_p, os.path.join(_arch, _fn)); print('dipindah:', _fn)

# [STAGE-1] CLIPS_PER_BATCH=16 / CLIP_LEN=1, CFG_TAG, path checkpoint bertag, assert payload unik, dan metadata cfg pada
# checkpoint SUDAH diterapkan di sel Config (2) dan sel training (6). Checkpoint lama (embedder tanh + extractor GAP2x2+FC)
# tidak kompatibel dan tidak akan termuat karena path checkpoint baru diberi tag CFG_TAG.


In [ ]:
# [AUDIT-PATCH] Sanity check (hard stop) + training penuh. Diagnostik Level 2 (6a) dijalankan sebelum sel ini.
bce = nn.BCEWithLogitsLoss()
CKPT_PATH = os.path.join(OUTPUT_DIR, 'models', f'wm_checkpoint_{CFG_TAG}.pth')   # [STAGE-1] bertag
BEST_PATH = os.path.join(OUTPUT_DIR, 'models', f'wm_best_{CFG_TAG}.pth')
CFG_SNAPSHOT = {'CLIPS_PER_BATCH': CLIPS_PER_BATCH, 'CLIP_LEN': CLIP_LEN, 'BPDA_CLIPS_PER_BATCH': BPDA_CLIPS_PER_BATCH,
                'BPDA_CLIP_LEN': BPDA_CLIP_LEN, 'MSG_BITS': MSG_BITS, 'EPOCHS': EPOCHS, 'STEPS_PER_EPOCH': STEPS_PER_EPOCH,
                'LAMBDA_DELTA': LAMBDA_DELTA, 'ALPHA_INIT': ALPHA_INIT, 'SEED': SEED, 'CFG_TAG': CFG_TAG,
                'NOTEBOOK_VERSION': NOTEBOOK_VERSION}
if not os.path.exists(CKPT_PATH):
    SANITY_ACC = sanity_check()

optimizer = torch.optim.Adam(list(embedder.parameters()) + list(extractor.parameters()), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.05)

history, start_epoch, lam_img, best_score = [], 1, LAMBDA_IMG_INIT, -1.0
if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    embedder.load_state_dict(ck['embedder']); extractor.load_state_dict(ck['extractor'])
    optimizer.load_state_dict(ck['optimizer']); scheduler.load_state_dict(ck['scheduler'])
    history, lam_img, best_score = ck['history'], ck['lam_img'], ck['best_score']
    start_epoch = ck['epoch'] + 1
    print(f'[RESUME] lanjut dari epoch {start_epoch}')

embedder.train(); extractor.train()
for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    lam_eff = lam_img * (0.0 if epoch <= WARMUP_EPOCHS else 1.0)
    stats = {'loss': 0, 'msg': 0, 'mse': 0, 'rate': 0, 'acc': 0, 'delta': 0}
    div = {'uniq_min': 10 ** 9, 'maxfp': 0, 'n': 0}     # bukti diversity step non-BPDA
    mode_count = {}
    for step in range(STEPS_PER_EPOCH):
        use_real = (epoch > WARMUP_EPOCHS) and (random.random() < REAL_CODEC_PROB)   # step BPDA/ffmpeg?
        n_clips, clip_len = (BPDA_CLIPS_PER_BATCH, BPDA_CLIP_LEN) if use_real else (CLIPS_PER_BATCH, CLIP_LEN)
        x = to_tensor(sample_clips(train_cache, n_clips, clip_len)).to(device)
        if random.random() < 0.5:
            x = torch.flip(x, dims=[3])
        msgs, msg_f = make_step_messages(n_clips, clip_len)   # payload independen per klip; assert unik
        _log_payloads(msgs)
        if not use_real:                                      # step tanpa ffmpeg: harus 16 payload unik, 1 frame/payload
            _u, _fp = payload_diversity(msg_f)
            div['uniq_min'] = min(div['uniq_min'], _u); div['maxfp'] = max(div['maxfp'], _fp); div['n'] += 1

        x_w, y, y_w = embed_tensor(x, msg_f)
        with torch.no_grad():
            x_c = codec.g_s(torch.round(y)).clamp(0, 1)
            rate0 = latent_rate_bits(y)
        x_att, mode = attack_layer(x_w, n_clips, epoch, use_real=use_real)
        mode_count[mode] = mode_count.get(mode, 0) + 1
        logits = extractor(x_att)

        loss_msg, loss_delta, loss_core = msg_delta_loss(logits, msg_f, y_w - y)   # [STAGE-1] bit + L2 delta
        loss_mse = F.mse_loss(x_w, x_c)
        loss_ssim = 1 - ssim_torch(x_w, x_c, data_range=1.0)
        loss_rate = F.relu(latent_rate_bits(y_w) - rate0).mean() / (x.shape[2] * x.shape[3])
        loss = loss_core + lam_eff * (MSE_SCALE * loss_mse + loss_ssim) + LAMBDA_RATE * loss_rate

        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(list(embedder.parameters()) + list(extractor.parameters()), 5.0)
        optimizer.step()

        stats['loss'] += loss.item(); stats['msg'] += loss_msg.item(); stats['mse'] += loss_mse.item()
        stats['rate'] += loss_rate.item(); stats['delta'] += loss_delta.item()
        stats['acc'] += ((logits > 0).float() == msg_f).float().mean().item()
    scheduler.step()
    with open(TRAIN_PAYLOADS_PATH, 'w') as _f:
        json.dump(sorted(TRAIN_PAYLOADS), _f)
    stats = {k: v / STEPS_PER_EPOCH for k, v in stats.items()}

    val = validate()
    if epoch > WARMUP_EPOCHS:   # anggaran imperceptibility adaptif
        lam_img = float(np.clip(lam_img * (1.25 if val['psnr_codec'] < TARGET_PSNR else 0.85), 0.05, 100.0))
    rec = {'epoch': epoch, 'lam_img': lam_eff, **{f'train_{k}': v for k, v in stats.items()}, **val,
           'uniq_payloads_per_step_min(nonBPDA)': div['uniq_min'] if div['n'] else None,
           'max_frames_per_payload(nonBPDA)': div['maxfp'] if div['n'] else None}
    history.append(rec)

    eligible = val['psnr_codec'] >= TARGET_PSNR - 1.0 or epoch <= WARMUP_EPOCHS
    if eligible and val['score'] > best_score and epoch > WARMUP_EPOCHS:
        best_score = val['score']
        torch.save({'embedder': embedder.state_dict(), 'extractor': extractor.state_dict(), 'epoch': epoch,
                    'val': val, 'cfg': CFG_SNAPSHOT}, BEST_PATH)
    torch.save({'embedder': embedder.state_dict(), 'extractor': extractor.state_dict(),
                'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
                'history': history, 'lam_img': lam_img, 'best_score': best_score, 'epoch': epoch, 'cfg': CFG_SNAPSHOT}, CKPT_PATH)

    acc_txt = ' '.join(f'{k[4:]}={v:.3f}' for k, v in val.items() if k.startswith('acc_'))
    print(f'Ep {epoch:03d} | {time.time()-t0:5.0f}s | loss={stats["loss"]:.3f} acc_train={stats["acc"]:.3f} delta_l2={stats["delta"]:.4f} | '
          f'uniq/step={div["uniq_min"] if div["n"] else "n/a"} frames/payload={div["maxfp"] if div["n"] else "n/a"} | '
          f'VAL PSNR(vs codec)={val["psnr_codec"]:.2f} SSIM={val["ssim_codec"]:.4f} | {acc_txt} | λimg={lam_eff:.2f}')

if not os.path.exists(BEST_PATH):   # misal QUICK_TEST: simpan model terakhir sebagai "best"
    torch.save({'embedder': embedder.state_dict(), 'extractor': extractor.state_dict(), 'epoch': EPOCHS,
                'val': history[-1] if history else {}, 'cfg': CFG_SNAPSHOT}, BEST_PATH)

hist_df = pd.DataFrame(history)
hist_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'), index=False)
if len(hist_df):
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(hist_df['epoch'], hist_df['train_loss']); ax[0].set_title('Train loss')
    for c in [c for c in hist_df.columns if c.startswith('acc_')]:
        ax[1].plot(hist_df['epoch'], hist_df[c], label=c[4:])
    ax[1].set_title('Bit accuracy frame-level (validasi)'); ax[1].legend(fontsize=8); ax[1].set_ylim(0.4, 1.02)
    ax[2].plot(hist_df['epoch'], hist_df['psnr_codec'], label='PSNR vs codec')
    ax[2].axhline(TARGET_PSNR, ls='--', c='g', label='target'); ax[2].legend(); ax[2].set_title('Imperceptibility')
    plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'training_curve.png'), dpi=130); plt.show()

## 7. Output 1 — Algoritma Watermarking (fungsi siap pakai)

| Fungsi | Peran |
|---|---|
| `embed_video(frames, msg, alpha)` | Input Video → **g_a** → **embedding latent** → entropy coding nyata → **g_s** → Video Output |
| `extract_logits(frames)` | video diterima → g_a → extractor → logit per frame |
| `phash_frames(frames)` | sidik jari konten per frame (dipakai validasi integritas) |
| registry (`registry.json`) | catatan setiap video yang di-watermark: ID, pemilik, payload, SHA-256, pHash per frame, metrik |

In [ ]:
_best = torch.load(BEST_PATH, map_location=device, weights_only=False)
embedder.load_state_dict(_best['embedder']); extractor.load_state_dict(_best['extractor'])
embedder.eval(); extractor.eval()
print(f"Model terbaik dimuat (epoch {_best['epoch']}).")

CHUNK = 16


@torch.no_grad()
def embed_video(frames_u8, msg_bits, alpha=ALPHA_INIT, with_reference=False):
    '''Output 1: embedding watermark di latent neural codec dengan bitstream NYATA.
    Return: frame watermarked (uint8), [frame codec tanpa watermark], rata-rata byte/frame (wm, ref).'''
    msg = torch.from_numpy(np.asarray(msg_bits, np.float32)).to(device)
    out_wm, out_ref, bytes_wm, bytes_ref = [], [], [], []
    for i in range(0, len(frames_u8), CHUNK):
        x = to_tensor(frames_u8[i:i + CHUNK]).to(device)
        y = codec.g_a(x)                                           # Neural Codec Encoder
        y_w, _ = embedder(y, msg.expand(len(x), -1), alpha)        # Watermark Embedding (latent)
        y_w_hat, nb = codec_code_latent(y_w)                       # kuantisasi + entropy coding
        out_wm.append(to_uint8(codec.g_s(y_w_hat)))                # Neural Codec Decoder
        bytes_wm += nb
        if with_reference:
            y_hat, nb0 = codec_code_latent(y)
            out_ref.append(to_uint8(codec.g_s(y_hat))); bytes_ref += nb0
    wm = np.concatenate(out_wm)
    ref = np.concatenate(out_ref) if with_reference else None
    return wm, ref, float(np.mean(bytes_wm)), (float(np.mean(bytes_ref)) if with_reference else None)


@torch.no_grad()
def extract_logits(frames_u8):
    outs = []
    for i in range(0, len(frames_u8), CHUNK):
        outs.append(extractor(to_tensor(frames_u8[i:i + CHUNK]).to(device)).cpu().numpy())
    return np.concatenate(outs)


def rescan_frames(frames_u8):
    return decode_message(extract_logits(frames_u8).mean(0))


def phash_frame(f):
    g = cv2.resize(cv2.cvtColor(f, cv2.COLOR_RGB2GRAY), (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(g)[:8, :8].flatten()
    bits = (d > np.median(d[1:])).astype(np.uint8)
    return int(''.join(map(str, bits)), 2)


def phash_frames(frames):
    return [phash_frame(f) for f in frames]


def hamming64(a, b):
    return bin(int(a) ^ int(b)).count('1')


def apply_attack(frames_u8, fps, attack):
    '''Serangan untuk evaluasi. Return (frame hasil, bpp file atau None).'''
    name, p = attack
    T, H, W, _ = frames_u8.shape
    if name == 'none':
        return frames_u8, None
    if name == 'neural':
        out = [to_uint8(codec_roundtrip(to_tensor(frames_u8[i:i + CHUNK]).to(device)))
               for i in range(0, T, CHUNK)]
        return np.concatenate(out), None
    if name == 'rescale':
        small = [cv2.resize(f, (int(W * p) // 2 * 2, int(H * p) // 2 * 2), interpolation=cv2.INTER_AREA) for f in frames_u8]
        return np.stack([cv2.resize(f, (W, H), interpolation=cv2.INTER_LINEAR) for f in small]), None
    out, size = ffmpeg_roundtrip(frames_u8, fps, name, p)
    return out, size * 8 / (T * H * W)


def load_registry():
    if os.path.exists(REGISTRY_PATH):
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return {}


def save_registry(reg):
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(reg, f, indent=1)


print('Fungsi Output 1 siap.')

## 7a. Freeze model (sebelum harness / final test)

Checkpoint sudah dipilih oleh **skor validasi** di bagian 6. Sel ini membekukan parameter dan mencatat hash; final test akan menolak berjalan bila hash berubah.

In [ ]:
for _m in (embedder, extractor):
    _m.eval()
    for _p in _m.parameters():
        _p.requires_grad_(False)

FROZEN_SHA = sha256_file(BEST_PATH)
FROZEN_INFO = {'best_path': BEST_PATH, 'sha256': FROZEN_SHA, 'epoch': int(_best['epoch']),
               'val_at_selection': {k: float(v) for k, v in (_best.get('val') or {}).items() if isinstance(v, (int, float))},
               'decision_threshold_logit': 0.0, 'alpha': ALPHA_INIT, 'retry': 'OFF'}
with open(os.path.join(OUTPUT_DIR, 'frozen_model.json'), 'w') as _f:
    json.dump(FROZEN_INFO, _f, indent=1)
print('MODEL DI-FREEZE:', {k: FROZEN_INFO[k] for k in ['epoch', 'sha256', 'decision_threshold_logit', 'alpha', 'retry']})

## 7e. [AUDIT v2 — Root cause Uji A] A0 vs A1, delta, extractor, message diversity, distribusi payload

Jalankan **setelah sel freeze 7a**. Memakai checkpoint beku yang sama, **tanpa training**, hanya `val_cache` (bukan video test). A0 = `y_w` langsung ke extractor (tanpa round); A1 = `round(y_w)` (sama dengan `L2_latent_quantized` di harness). Ada kontrol tanpa watermark. Kirim output sel + `diag_root_cause.json` + `diag_perbit.csv`.

In [ ]:
# ===== AUDIT v2 -- Root cause Uji A: A0 (tanpa round) vs A1 (round) pada checkpoint BEKU =====
# Tanpa training, tanpa optimizer, tanpa video test (hanya val_cache). Model tidak diubah.
import os, json
import numpy as np, pandas as pd, torch

assert not any(p.requires_grad for p in embedder.parameters()) and not any(p.requires_grad for p in extractor.parameters()), \
    'Jalankan sel freeze 7a dulu (model harus beku)'
embedder.eval(); extractor.eval()

D_FRAMES_PER_VIDEO, D_CHUNK, D_SEED = 24, 16, 2026
ID_SL, CRC_SL = slice(0, 32), slice(32, 48)
DIAG = {}


def _draw_msgs(kind, n, rng):
    if kind == 'raw':                                   # distribusi yang dipakai training (48 bit acak murni)
        return rng.randint(0, 2, (n, MSG_BITS)).astype(np.float32)
    return np.stack([build_message(int(rng.randint(0, 2 ** 32, dtype=np.uint64))) for _ in range(n)]).astype(np.float32)


@torch.no_grad()
def d_read(y):                                          # identik dengan logits_from_latent() di harness
    return extractor.read_latent(y)


@torch.no_grad()
def d_logits(Y, fn=lambda t: t):
    return torch.cat([d_read(fn(Y[i:i + D_CHUNK].to(device))).cpu() for i in range(0, len(Y), D_CHUNK)])


@torch.no_grad()
def collect(kind, alpha=ALPHA_INIT):
    rng = np.random.RandomState(D_SEED + (0 if kind == 'raw' else 1))
    out = {k: [] for k in ['y', 'yw', 'm', 'yw_alt', 'm_alt']}
    for name in sorted(val_cache):
        fr = val_cache[name][:D_FRAMES_PER_VIDEO]
        for i in range(0, len(fr), D_CHUNK):
            x = to_tensor(fr[i:i + D_CHUNK]).to(device); n = len(x)
            m = torch.from_numpy(_draw_msgs(kind, n, rng)).to(device)       # tiap frame pesannya SENDIRI (tanpa voting)
            m2 = torch.from_numpy(_draw_msgs(kind, n, rng)).to(device)      # pesan kedua untuk tes ketergantungan-pesan
            y = codec.g_a(x)
            yw, _ = embedder(y, m, ALPHA_INIT); yw2, _ = embedder(y, m2, ALPHA_INIT)
            for k, v in zip(out, [y, yw, m, yw2, m2]):
                out[k].append(v.cpu())
    return {k: torch.cat(v) for k, v in out.items()}


def d_metrics(logits, m):
    pred = (logits > 0).int().numpy(); gt = m.int().numpy(); ok = pred == gt
    return ({'bit_acc': float(ok.mean()), 'exact': float(ok.all(1).mean()), 'id_acc': float(ok[:, ID_SL].mean()),
             'crc_acc': float(ok[:, CRC_SL].mean()), 'n_frames': len(gt)}, ok.mean(0), pred)


# ---------------------------------------------------------------- 1) A0 vs A1 (+ kontrol tanpa watermark)
rows, PB = [], {}
DATA = {}
for kind in ['raw', 'id_crc']:
    D = collect(kind); DATA[kind] = D
    for exp, rnd, Ysrc in [('A0', 'No', D['yw']), ('A1', 'Yes', torch.round(D['yw'])),
                           ('CTRL (y asli, tanpa WM)', 'No', D['y']), ('CTRL (round(y), tanpa WM)', 'Yes', torch.round(D['y']))]:
        lg = d_logits(Ysrc); met, pb, pred = d_metrics(lg, D['m'])
        rows.append({'Payload': kind, 'Experiment': exp, 'Round': rnd, 'Bit Accuracy': round(met['bit_acc'], 4),
                     'Exact Match': round(met['exact'], 4), 'ID Accuracy': round(met['id_acc'], 4),
                     'CRC Accuracy': round(met['crc_acc'], 4), 'N frame': met['n_frames']})
        PB[(kind, exp)] = (pb, lg, pred)
T_A = pd.DataFrame(rows)
print('=== A0 vs A1 (checkpoint beku, video validasi, 1 pesan unik per frame, tanpa voting) ===')
print(T_A.to_string(index=False))
DIAG['A0_A1'] = rows

# rekonsiliasi dengan harness (soft-voting per video, 1 pesan/video)
_csv = os.path.join(OUTPUT_DIR, 'exp_stage1_validation_rows.csv')
if os.path.exists(_csv):
    _h = pd.read_csv(_csv); _h = _h[_h['video_role'] == 'val_video']
    print('\n[Rekonsiliasi harness, val_video, soft-voting] ',
          {c: round(float(_h[_h['condition'] == c]['bit_acc'].mean()), 4) for c in ['L2_latent_float', 'L2_latent_quantized']})

# ---------------------------------------------------------------- 2) Audit delta embedding
D = DATA['raw']; y, yw = D['y'], D['yw']
d = (yw - y); ad = d.abs().flatten().numpy()
p50, p90, p95, p99 = np.percentile(ad, [50, 90, 95, 99])
d_alt = (D['yw_alt'] - y)
DELTA = {'mean_abs_delta': float(ad.mean()), 'max_abs_delta': float(ad.max()), 'std_delta': float(d.std()),
         'p50': float(p50), 'p90': float(p90), 'p95': float(p95), 'p99': float(p99),
         'frac_abs_delta_gt_0.5': float((ad > 0.5).mean()),
         'frac_round(yw)!=round(y)': float((torch.round(yw) != torch.round(y)).float().mean()),
         'frac_round(y)==0': float((torch.round(y) == 0).float().mean()),
         'frac_round(yw)==0': float((torch.round(yw) == 0).float().mean()),
         'y_std': float(y.std()), 'LATENT_SCALE': float(LATENT_SCALE), 'DELTA_MAX': float(DELTA_MAX),
         # ketergantungan pada PESAN: bandingkan delta untuk 2 pesan berbeda pada frame yang SAMA
         'mean|delta(m1)-delta(m2)| / mean|delta|': float((d - d_alt).abs().mean() / d.abs().mean().clamp_min(1e-12)),
         'frac_round(yw_m1)!=round(yw_m2)': float((torch.round(yw) != torch.round(D['yw_alt'])).float().mean())}
print('\n=== Audit delta = y_w - y (satuan = langkah kuantisasi; N elemen = %d) ===' % len(ad))
for k, v in DELTA.items():
    print(f'  {k:<42s} {v:.5f}')
DIAG['delta'] = DELTA

# ---------------------------------------------------------------- 3) Audit extractor (per bit)
pb0, _, _ = PB[('raw', 'A0')]; pb1, lg1, pred1 = PB[('raw', 'A1')]
prob1 = torch.sigmoid(lg1).numpy(); gt1 = D['m'].numpy()
T_B = pd.DataFrame({'Bit': range(MSG_BITS), 'Type': ['ID'] * 32 + ['CRC'] * 16,
                    'Acc_A0': pb0.round(3), 'Acc_A1': pb1.round(3),
                    'MeanProb_A1': prob1.mean(0).round(3), 'PredOnes_A1': pred1.mean(0).round(3),
                    'LogitStd_A1': lg1.numpy().std(0).round(3), 'GT_ones': gt1.mean(0).round(3)})
print('\n=== Per-bit (payload acak, 48 posisi; Type = posisi ID/CRC menurut format, bukan isi) ===')
print(T_B.to_string(index=False))
T_B.to_csv(os.path.join(OUTPUT_DIR, 'diag_perbit.csv'), index=False)
EXT = {'n_bits_acc_gt_0.7_A1': int((pb1 > 0.7).sum()), 'n_bits_acc_gt_0.9_A1': int((pb1 > 0.9).sum()),
       'n_bits_acc_gt_0.7_A0': int((pb0 > 0.7).sum()), 'n_bits_acc_gt_0.9_A0': int((pb0 > 0.9).sum()),
       'max_bit_acc_A1': float(pb1.max()), 'min_bit_acc_A1': float(pb1.min()),
       'n_collapsed_bits(predOnes<0.05 or >0.95)': int(((pred1.mean(0) < 0.05) | (pred1.mean(0) > 0.95)).sum()),
       'mean_logit_std_per_bit': float(lg1.numpy().std(0).mean())}
print('\nRingkas extractor:', json.dumps(EXT, indent=1))
DIAG['extractor'] = EXT

# ---------------------------------------------------------------- 4) Audit message diversity (kode aktual + simulasi persis)
print('\n=== Message diversity ===')
print(f'CLIPS_PER_BATCH={CLIPS_PER_BATCH}  CLIP_LEN={CLIP_LEN}  frame/step={CLIPS_PER_BATCH * CLIP_LEN}')
try:                                                   # cari baris training yang benar-benar dieksekusi di sesi ini
    _hist = get_ipython().user_ns.get('In', [])
    _hit = [l.strip() for s in _hist if 'repeat_interleave(CLIP_LEN' in s for l in s.splitlines()
            if 'randint(0, 2, (CLIPS_PER_BATCH' in l or 'repeat_interleave' in l]
    print('Baris training terdeteksi di sesi ini:', sorted(set(_hit)) or 'tidak ada (sel training tidak dijalankan di sesi ini)')
except Exception as e:
    print('Riwayat sel tidak tersedia:', repr(e))
_u = []
for _ in range(200):                                    # simulasi persis dua baris training: randint + repeat_interleave
    _m = torch.randint(0, 2, (CLIPS_PER_BATCH, MSG_BITS)).float(); _mf = _m.repeat_interleave(CLIP_LEN, 0)
    _u.append(len({tuple(r) for r in _mf.int().tolist()}))
print(f'Payload unik per step (200 step simulasi): min={min(_u)} max={max(_u)}  | frame per payload = {CLIP_LEN}')
DIAG['diversity'] = {'CLIPS_PER_BATCH': CLIPS_PER_BATCH, 'CLIP_LEN': CLIP_LEN, 'unique_per_step_min': min(_u), 'unique_per_step_max': max(_u)}

# ---------------------------------------------------------------- 5) Audit distribusi payload
_tr = torch.randint(0, 2, (20000, MSG_BITS)).float()    # generator yang sama dgn training
_rng = np.random.RandomState(7)
_id = _draw_msgs('id_crc', 5000, _rng)
T_P = pd.DataFrame({'Bit': range(MSG_BITS), 'Type': ['ID'] * 32 + ['CRC'] * 16,
                    'P0_train_gen': (1 - _tr.mean(0).numpy()).round(3), 'P1_train_gen': _tr.mean(0).numpy().round(3),
                    'P0_id_crc': (1 - _id.mean(0)).round(3), 'P1_id_crc': _id.mean(0).round(3)})
print('\n=== Distribusi payload (train generator n=20000 | ID+CRC n=5000) ===')
print(T_P.to_string(index=False))
print(f'max|P1-0.5|: train_gen={np.abs(_tr.mean(0).numpy() - 0.5).max():.3f}  id_crc={np.abs(_id.mean(0) - 0.5).max():.3f} '
      f'| payload unik train_gen: {len({tuple(r) for r in _tr.int().tolist()})}/20000')
print('CRC-16 = fungsi deterministik dari ID 32 bit: 16 bit CRC tidak membawa informasi independen (dibuktikan oleh build_message).')
print('Ukuran TRAIN_PAYLOADS (log payload training):', len(globals().get('TRAIN_PAYLOADS', [])), '-> 0 berarti "unseen" tidak dapat diverifikasi')
DIAG['payload'] = {'max_dev_train_gen': float(np.abs(_tr.mean(0).numpy() - 0.5).max()), 'max_dev_id_crc': float(np.abs(_id.mean(0) - 0.5).max()),
                   'TRAIN_PAYLOADS_size': len(globals().get('TRAIN_PAYLOADS', []))}

with open(os.path.join(OUTPUT_DIR, 'diag_root_cause.json'), 'w') as _f:
    json.dump(DIAG, _f, indent=1)
print('\nTersimpan: diag_root_cause.json, diag_perbit.csv (kirim kedua file + output sel ini).')


## 7b. Harness EXP-01..07 (bit-level, first-pass, tanpa retry)

**Tiga level**
1. **Level 1** payload encoding: `text -> bits -> CRC -> bits` (EXP-01, tanpa GPU).
2. **Level 2** ekstraksi tanpa serangan tambahan. Empat kondisi baca:
   `L2_latent_float` (tanpa codec) · `L2_latent_quantized` (hanya kuantisasi) · `L2_video_codec1x` (video watermarked hasil g_s + bitstream nyata, tanpa serangan) .
3. **Level 3** `L3_video_codec2x`: video watermarked **dilewatkan lagi** ke Latent Neural Codec lalu didecode.

Catatan: label `none` di evaluasi lama = `L2_video_codec1x`, yang sudah melewati codec sekali (embedding terjadi di dalam codec). Baca latent langsung adalah satu-satunya baca yang benar-benar tanpa codec.

**Protokol:** Tahap 1 (validasi) menjalankan Level 1-3 pada video val + train dan **gerbang Level 2** (`GATE_BITACC`, `GATE_EXACT`, ditetapkan sebelum melihat hasil). Jika gerbang gagal, **final test tidak dijalankan**. Tahap 2 (final test) hanya sekali, model frozen, threshold logit 0, retry OFF.

`raw` = bit hasil threshold logit rata-rata antar frame (sebelum CRC). `ecc_decode` = verifikasi CRC-16 (`ECC_REPEAT=1`, tidak ada koreksi error).

In [ ]:
GATE_BITACC = 0.95      # ditetapkan a priori; dievaluasi pada VALIDASI, bukan test
GATE_EXACT  = 0.90
MAX_HARNESS_FRAMES = 8 if QUICK_TEST else 24
N_HARNESS_VIDEOS = {'train': 5, 'test': 10}
RUN_FINAL_TEST = True   # tetap dijaga gerbang Level 2 dan flag sekali-jalan
assert ECC_REPEAT == 1, 'harness ditulis untuk ECC_REPEAT=1 (tanpa repetition code)'


# ---------------- Level 1: payload ----------------
def text4_to_id(t):
    b = t.encode('ascii'); assert len(b) == ID_BITS // 8, 'teks harus 4 karakter ASCII'
    return int.from_bytes(b, 'big')


def id_to_text(i):
    b = int(i).to_bytes(ID_BITS // 8, 'big')
    return b.decode('ascii') if all(32 <= c < 127 for c in b) else None


def bits_to_int_(b):
    return int(''.join(map(str, np.asarray(b).astype(int))), 2)


def exp01_payload_roundtrip(n_random=1000):
    """EXP-01: text -> bits -> CRC -> bits -> decode (logit sempurna). Harus lulus sebelum apa pun."""
    cases = [('zeros', 0), ('ones', 2 ** 32 - 1), ('1010', 0xAAAAAAAA), ('0101', 0x55555555), ('text_budi', text4_to_id('budi'))]
    for name, i in cases:
        m = build_message(i); d = decode_message((m * 2 - 1) * 5)
        assert d['id'] == i and d['crc_ok'] and bit_error_rate(d['bits'], m[:PAYLOAD_BITS]) == 0, f'EXP-01 gagal: {name}'
    assert id_to_text(text4_to_id('budi')) == 'budi'
    rng = np.random.RandomState(1)
    for _ in range(n_random):
        i = int(rng.randint(0, 2 ** 32, dtype=np.uint64)); m = build_message(i); d = decode_message((m * 2 - 1) * 3)
        assert d['id'] == i and d['crc_ok'] and bit_error_rate(d['bits'], m[:PAYLOAD_BITS]) == 0
    print(f'EXP-01 LULUS: {len(cases)} kasus pola/teks + {n_random} ID acak (round-trip exact).')


def make_payload_sets(seed=12345):
    """Semua payload berupa pesan MSG_BITS (=48) bit. 'gt' = bit ground truth yang dibandingkan."""
    rng = np.random.RandomState(seed); P = []

    def add(name, role, kind, msg, crc, pid=None):
        msg = np.asarray(msg).astype(int); assert len(msg) == MSG_BITS
        P.append(dict(name=name, role=role, kind=kind, msg=msg, gt=msg[:PAYLOAD_BITS], crc=crc, id=pid))

    raws = {'zeros': np.zeros(MSG_BITS, int), 'ones': np.ones(MSG_BITS, int),
            '1010': np.tile([1, 0], MSG_BITS // 2), '0101': np.tile([0, 1], MSG_BITS // 2)}
    for n, b in raws.items():
        add(f'raw_{n}', 'unseen_pattern', 'pattern_raw', b, False)
    for t in ['budi', 'kamu']:
        i = text4_to_id(t); add(f'text_{t}', 'unseen_text', 'text_id_crc', build_message(i), True, i)
    i = make_watermark_id(OWNER_TEXT, 'fixed_probe.mp4', '2026-01-01T00:00:00')
    add('text_sabila_hmac', 'unseen_text', 'text_hmac_crc', build_message(i), True, i)
    for k in range(6):
        i = int(rng.randint(0, 2 ** 32, dtype=np.uint64)); add(f'rand_id_{k}', 'unseen_random', 'random_id_crc', build_message(i), True, i)
    for k in range(2):
        add(f'rand_raw_{k}', 'unseen_random', 'random_raw', rng.randint(0, 2, MSG_BITS), False)
    rv = np.random.RandomState(0)                    # persis payload validate(): satu draw per video validasi
    for k in range(len(val_cache)):
        add(f'val_payload_{k}', 'val_payload', 'random_raw', rv.randint(0, 2, MSG_BITS), False)
    tr = sorted(TRAIN_PAYLOADS)
    for k, v in enumerate(tr[::max(1, len(tr) // 3)][:3]):
        add(f'train_payload_{k}', 'train_payload', 'random_raw', [(v >> (MSG_BITS - 1 - j)) & 1 for j in range(MSG_BITS)], False)
    return P


def assert_unseen(P):
    if not TRAIN_PAYLOADS:
        print('[PERINGATAN] log payload training kosong (checkpoint lama?). Status "unseen" TIDAK dapat diverifikasi.')
        return False
    val_set = {bits_to_int_(p['msg']) for p in P if p['role'] == 'val_payload'}
    for p in P:
        if p['role'].startswith('unseen'):
            v = bits_to_int_(p['msg'])
            assert v not in TRAIN_PAYLOADS and v not in val_set, f"payload {p['name']} pernah dipakai training/validasi"
    print(f'Unseen terverifikasi: {sum(p["role"].startswith("unseen") for p in P)} payload tidak ada di {len(TRAIN_PAYLOADS)} payload training / validasi.')
    return True


# ---------------- Metrik bit ----------------
def compare_bits(pred, gt):
    pred = np.asarray(pred).astype(int).ravel(); gt = np.asarray(gt).astype(int).ravel()
    mism = len(pred) != len(gt); n = min(len(pred), len(gt))
    if mism:
        print(f'[PERINGATAN] panjang bit berbeda: pred={len(pred)} gt={len(gt)} -> metrik dihitung pada {n} bit pertama')
    acc = float(np.mean(pred[:n] == gt[:n]))
    return {'bit_acc': acc, 'ber': 1.0 - acc, 'len_pred': len(pred), 'len_gt': len(gt), 'length_mismatch': bool(mism),
            'exact_match': bool((not mism) and np.all(pred == gt))}


def decode_report(mean_logits, p):
    dec = decode_message(mean_logits); pred = dec['bits']
    r = compare_bits(pred, p['gt'])
    r['pred_ones_frac'] = float(np.mean(pred)); r['pred_bits'] = ''.join(map(str, pred)); r['gt_bits'] = ''.join(map(str, p['gt']))
    if p['crc']:
        r['crc_ok'] = bool(dec['crc_ok'])
        r['ecc_decode'] = 'SUCCESS' if (dec['crc_ok'] and dec['id'] == p['id']) else 'FAIL'
        r['decoded_payload'] = dec['id_hex'] if dec['crc_ok'] else None
        r['decoded_text'] = id_to_text(dec['id']) if dec['crc_ok'] else None
    else:
        r.update(crc_ok=None, ecc_decode='N/A', decoded_payload=None, decoded_text=None)
    return r


def aggregate(df, by):
    agg = df.groupby(by).agg(n=('bit_acc', 'size'), mean_bit_acc=('bit_acc', 'mean'), median_bit_acc=('bit_acc', 'median'),
                             worst_bit_acc=('bit_acc', 'min'), mean_ber=('ber', 'mean'),
                             exact_match_rate=('exact_match', 'mean'), pred_ones_frac=('pred_ones_frac', 'mean'))
    crc_df = df[df['ecc_decode'] != 'N/A']
    if len(crc_df):
        agg = agg.join(crc_df.groupby(by)['ecc_decode'].apply(lambda s: float((s == 'SUCCESS').mean())).rename('ecc_success_rate'))
    return agg


def table_E(df):
    t = df[['video', 'payload', 'payload_role', 'condition', 'bit_acc', 'ber', 'ecc_decode', 'exact_match']].copy()
    return t.rename(columns={'video': 'Video', 'payload': 'Payload', 'payload_role': 'Payload Type', 'condition': 'Codec/Kondisi',
                             'bit_acc': 'Bit Accuracy', 'ber': 'BER', 'ecc_decode': 'ECC Decode', 'exact_match': 'Exact Match'})


def level2_gate(df):
    d = df[(df['condition'] == 'L2_video_codec1x') & (df['video_role'] == 'val_video')]
    acc, em = float(d['bit_acc'].mean()), float(d['exact_match'].mean())
    return bool(acc >= GATE_BITACC and em >= GATE_EXACT), acc, em


# ---------------- Embedding + 4 kondisi baca ----------------
@torch.no_grad()
def embed_latents(frames_u8, msg_bits, alpha=ALPHA_INIT):
    msg = torch.from_numpy(np.asarray(msg_bits, np.float32)).to(device)
    yw, xs = [], []
    for i in range(0, len(frames_u8), CHUNK):
        x = to_tensor(frames_u8[i:i + CHUNK]).to(device)
        y = codec.g_a(x)
        y_w, _ = embedder(y, msg.expand(len(x), -1), alpha)
        y_w_hat, _ = codec_code_latent(y_w)               # kuantisasi + entropy coding NYATA
        yw.append(y_w); xs.append(to_uint8(codec.g_s(y_w_hat)))
    return torch.cat(yw), np.concatenate(xs)


@torch.no_grad()
def logits_from_latent(y):
    return extractor.read_latent(y).cpu().numpy()


@torch.no_grad()
def read_conditions(y_w, x_w_u8):
    out = {'L2_latent_float': logits_from_latent(y_w),
           'L2_latent_quantized': logits_from_latent(torch.round(y_w)),
           'L2_video_codec1x': extract_logits(x_w_u8)}
    xa = np.concatenate([to_uint8(codec_roundtrip(to_tensor(x_w_u8[i:i + CHUNK]).to(device))) for i in range(0, len(x_w_u8), CHUNK)])
    out['L3_video_codec2x'] = extract_logits(xa)
    return out


def load_frames(names, n):
    return {v: read_video(os.path.join(DATASET_DIR, v), max_frames=n)[0] for v in names}


def run_harness(video_groups, payloads, stage):
    rows = []
    for vrole, vids in video_groups.items():
        for v, frames in vids.items():
            for p in payloads:                                # first-pass tunggal, alpha=ALPHA_INIT, tanpa retry
                y_w, x_w = embed_latents(frames, p['msg'])
                for cond, lg in read_conditions(y_w, x_w).items():
                    r = decode_report(lg.mean(0), p)
                    r['bit_acc_frame_mean'] = float(np.mean((lg > 0).astype(int) == p['gt'][None]))
                    rows.append({'stage': stage, 'video': v, 'video_role': vrole, 'payload': p['name'], 'payload_role': p['role'],
                                 'payload_kind': p['kind'], 'condition': cond, **r})
    return pd.DataFrame(rows)


def show_summary(df, tag):
    print(f'\n===== {tag}: agregat per kondisi x peran video x peran payload =====')
    display(aggregate(df, ['condition', 'video_role', 'payload_role']).round(4))
    print(f'\n===== {tag}: pola biner (deteksi extractor bias/konstan) =====')
    display(aggregate(df[df['payload_kind'] == 'pattern_raw'], ['condition', 'payload']).round(4))


def run_stage1_validation():
    P = make_payload_sets(); assert_unseen(P)
    vids = {'val_video': load_frames(val_videos, MAX_HARNESS_FRAMES),
            'train_video': load_frames(train_videos[:N_HARNESS_VIDEOS['train']], MAX_HARNESS_FRAMES)}
    df = run_harness(vids, P, 'validation')
    df.to_csv(os.path.join(OUTPUT_DIR, 'exp_stage1_validation_rows.csv'), index=False)
    show_summary(df, 'TAHAP 1 (VALIDASI)')
    ok, acc, em = level2_gate(df)
    print(f'\nGERBANG LEVEL 2 (val_video, L2_video_codec1x): bit_acc={acc:.4f} (>= {GATE_BITACC}) | exact_match={em:.4f} (>= {GATE_EXACT}) -> {"LULUS" if ok else "GAGAL"}')
    return df, ok


def run_stage2_final_test(force=False):
    assert sha256_file(BEST_PATH) == FROZEN_SHA, 'Model berubah sejak freeze. Final test dibatalkan.'
    flag = os.path.join(OUTPUT_DIR, 'final_test_done.flag')
    if os.path.exists(flag) and not force:
        raise RuntimeError('Final test sudah pernah dijalankan. Jangan diulang untuk tuning (force=True hanya untuk replikasi identik).')
    P = make_payload_sets(); assert_unseen(P)
    vids = {'test_video': load_frames(eval_videos[:N_HARNESS_VIDEOS['test']], MAX_HARNESS_FRAMES),
            'train_video': load_frames(train_videos[:N_HARNESS_VIDEOS['train']], MAX_HARNESS_FRAMES)}
    df = run_harness(vids, P, 'final_test')
    df.to_csv(os.path.join(OUTPUT_DIR, 'exp_final_test_rows.csv'), index=False)
    table_E(df).to_csv(os.path.join(OUTPUT_DIR, 'exp_final_test_tableE.csv'), index=False)
    show_summary(df, 'FINAL TEST')
    for role, exp in [('train_video', 'EXP-05 (video seen, payload unseen)'), ('test_video', 'EXP-06/07 (video unseen; payload seen atau unseen)')]:
        print(f'\n--- {exp} ---')
        display(aggregate(df[df['video_role'] == role], ['condition', 'payload_role']).round(4))
    open(flag, 'w').write(FROZEN_SHA)
    return df

## 7c. Jalankan harness

1. EXP-01 (Level 1) harus lulus.
2. Tahap 1 pada validasi. Jika gerbang Level 2 gagal, **berhenti**: perbaiki Level 2 (lihat Diagnostik 6a) dan train ulang. Jangan menyentuh final test.
3. Tahap 2 (final test) hanya jika gerbang lulus; model frozen; sekali jalan.

In [ ]:
exp01_payload_roundtrip()
df_val, LEVEL2_OK = run_stage1_validation()

if not LEVEL2_OK:
    print('\nSTOP: Level 2 GAGAL pada VALIDASI. Final test TIDAK dijalankan.')
    print('Jangan menyimpulkan apa pun tentang Latent Neural Codec (Level 3) sebelum Level 2 lulus.')
    df_final = None
elif RUN_FINAL_TEST:
    df_final = run_stage2_final_test()
    print('\nFinal test selesai. Tabel per video x payload: exp_final_test_tableE.csv')
    display(table_E(df_final).head(30))

## 7d. [AUDIT v2 — Aksi 3] Gerbang sesuai customer + akurasi per posisi bit

Tanpa GPU. Membaca `exp_stage1_validation_rows.csv` dari harness (7c). Gerbang Uji A memakai `L2_latent_quantized` (>0,98), Uji B memakai `L2_video_codec1x` (>0,95), dengan metrik soft-voting dan per-frame. Analisis per bit memisahkan bit ID (0–31) dan CRC (32–47).

In [ ]:
import os, json
import numpy as np
import pandas as pd

TARGET_A, TARGET_B = 0.98, 0.95          # threshold customer, TIDAK diubah


def per_bit_analysis(csv_path, condition, video_role='val_video'):
    """Akurasi per posisi bit dari kolom pred_bits/gt_bits (payload acak vs pola konstan)."""
    df = pd.read_csv(csv_path, dtype={'pred_bits': str, 'gt_bits': str})
    d = df[(df['condition'] == condition) & (df['video_role'] == video_role)]
    out = {}
    for name, sub in [('acak (random_id_crc+random_raw)', d[d['payload_kind'].isin(['random_id_crc', 'random_raw'])]),
                      ('pola konstan (pattern_raw)', d[d['payload_kind'] == 'pattern_raw'])]:
        if not len(sub):
            continue
        P = np.array([[int(c) for c in s] for s in sub['pred_bits']])
        G = np.array([[int(c) for c in s] for s in sub['gt_bits']])
        acc = (P == G).mean(0)
        out[name] = {'n': len(sub), 'acc_per_bit': acc,
                     'acc_ID_bit0_31': float(acc[:32].mean()), 'acc_CRC_bit32_47': float(acc[32:].mean()),
                     'bit_terbaik': int(acc.argmax()), 'acc_terbaik': float(acc.max()),
                     'bit_terburuk': int(acc.argmin()), 'acc_terburuk': float(acc.min()),
                     'n_bit_>0.7': int((acc > 0.7).sum())}
    return out


def gate_customer(csv_path, video_role='val_video'):
    """Uji A: L2_latent_quantized >0.98. Uji B: L2_video_codec1x >0.95.
    Dua metrik: soft-voting antar frame (bit_acc) dan per-frame tanpa voting (bit_acc_frame_mean)."""
    df = pd.read_csv(csv_path)
    df = df[df['video_role'] == video_role]
    rows = []
    for uji, cond, thr in [('Uji A', 'L2_latent_quantized', TARGET_A), ('Uji B', 'L2_video_codec1x', TARGET_B)]:
        d = df[df['condition'] == cond]
        for metrik in ['bit_acc', 'bit_acc_frame_mean']:
            v = float(d[metrik].mean())
            rows.append({'uji': uji, 'kondisi': cond, 'metrik': metrik, 'n': len(d), 'hasil': round(v, 4),
                         'target': f'>{thr}', 'status': 'PASS' if v > thr else 'FAIL'})
    return pd.DataFrame(rows)


if __name__ == '__main__' or 'OUTPUT_DIR' in globals():
    _csv = os.path.join(globals().get('OUTPUT_DIR', '.'), 'exp_stage1_validation_rows.csv')
    if os.path.exists(_csv):
        print(gate_customer(_csv).to_string(index=False))
        for cond in ['L2_latent_quantized', 'L2_video_codec1x']:
            for k, r in per_bit_analysis(_csv, cond).items():
                print(f'\n[{cond}] {k}: n={r["n"]} | ID(0-31)={r["acc_ID_bit0_31"]:.3f} CRC(32-47)={r["acc_CRC_bit32_47"]:.3f} '
                      f'| terbaik bit{r["bit_terbaik"]}={r["acc_terbaik"]:.3f} terburuk bit{r["bit_terburuk"]}={r["acc_terburuk"]:.3f} '
                      f'| bit>0.7: {r["n_bit_>0.7"]}/48')
    else:
        print('CSV tidak ditemukan:', _csv)

## 8. Loop Flowchart Pembimbing pada Data Uji

| Fungsi | Kotak flowchart |
|---|---|
| `insert_watermark(...)` | Insert Watermark (embedding latent neural codec) |
| `compress_video(...)` | Compress Video — **HEVC CRF 35** (`LOOP_ATTACK`), sesuai latar belakang riset |
| `rescan_watermark(...)` | Rescan Watermark |
| `is_legible(...)` | CRC valid **dan** ID sama dengan yang disisipkan |
| `run_watermark_pipeline(...)` | loop retry (α dinaikkan, maks `MAX_RETRY`) + pendaftaran ke registry |

Pada versi lama, kotak *Compress Video* memakai neural codec yang sama dengan yang dilatihkan, sehingga pengujian terlalu mudah. Karena neural codec sekarang menjadi **domain penyisipan**, kotak kompresi diisi codec konvensional berkompresi tinggi yang memang menjadi masalah pada latar belakang riset.

In [ ]:
registry = load_registry()


def insert_watermark(video_name, frames, fps, wm_id, alpha):
    '''[Insert Watermark]'''
    msg = build_message(wm_id)
    wm, ref, b_wm, b_ref = embed_video(frames, msg, alpha, with_reference=True)
    path = os.path.join(OUTPUT_DIR, 'watermarked', os.path.splitext(video_name)[0] + '_wm.mkv')
    write_video(wm, path, fps, codec='ffv1')
    return path, wm, ref, b_wm, b_ref


def compress_video(path, video_name, fps, attack=LOOP_ATTACK):
    '''[Compress Video] HEVC CRF tinggi (default).'''
    frames, _, _ = read_video(path, max_frames=None)
    name, crf = attack
    out = os.path.join(OUTPUT_DIR, 'compressed', f'{os.path.splitext(video_name)[0]}_wm_{name}_crf{crf}{codec_ext(name)}')
    write_video(frames, out, fps, codec=name, crf=crf)
    return out


def rescan_watermark(path):
    '''[Rescan Watermark]'''
    frames, _, _ = read_video(path, max_frames=None)
    return rescan_frames(frames)


def is_legible(dec, wm_id):
    '''[is legible Watermark?]'''
    return dec['crc_ok'] and dec['id'] == wm_id


def run_watermark_pipeline(video_name, max_retry=MAX_RETRY, verbose=True):
    frames, fps, orig_size = read_video(os.path.join(DATASET_DIR, video_name))
    created = datetime.datetime.now().isoformat(timespec='seconds')
    wm_id = make_watermark_id(OWNER_TEXT, video_name, created)
    for attempt in range(1, max_retry + 1):
        alpha = ALPHA_INIT * (1 + RETRY_ALPHA_STEP * (attempt - 1))
        wm_path, wm, ref, b_wm, b_ref = insert_watermark(video_name, frames, fps, wm_id, alpha)
        comp_path = compress_video(wm_path, video_name, fps)
        dec = rescan_watermark(comp_path)
        ok = is_legible(dec, wm_id)
        if verbose:
            print(f'[{video_name}] percobaan {attempt}/{max_retry} α={alpha:.2f} -> ID terbaca {dec["id_hex"]} '
                  f'CRC={dec["crc_ok"]} conf={dec["confidence"]:.3f} -> {"LEGIBLE" if ok else "tidak legible"}')
        if ok:
            break
    hw = FRAME_SIZE[0] * FRAME_SIZE[1]
    record = {
        'wm_id': f'{wm_id:08x}', 'owner': OWNER_TEXT, 'video_name': video_name, 'created_at': created,
        'payload_bits': ''.join(map(str, build_message(wm_id).astype(int))), 'alpha': alpha,
        'attempts': attempt, 'legible': bool(ok), 'n_frames': int(len(wm)), 'fps': fps,
        'frame_size': list(FRAME_SIZE), 'orig_size': list(orig_size),
        'sha256_known': [sha256_file(wm_path), sha256_file(comp_path)],
        'phash': [f'{h:016x}' for h in phash_frames(wm)],
        'bpp_wm': b_wm * 8 / hw, 'bpp_codec': b_ref * 8 / hw,
        'watermarked_path': wm_path, 'compressed_path': comp_path,
    }
    if ok:
        registry[record['wm_id']] = record
        save_registry(registry)
    return record, frames, wm, ref


pipeline_rows, eval_store = [], {}
for v in eval_videos:
    rec, frames, wm, ref = run_watermark_pipeline(v, max_retry=EVAL_MAX_RETRY)
    eval_store[v] = {'orig': frames, 'wm': wm, 'ref': ref, 'record': rec}
    pipeline_rows.append({k: rec[k] for k in ['video_name', 'wm_id', 'legible', 'attempts', 'alpha', 'bpp_codec', 'bpp_wm']})
    if rec['legible']:
        print(f'    -> print(True, "{rec["payload_bits"]}")')

df_pipeline = pd.DataFrame(pipeline_rows)
df_pipeline.to_csv(os.path.join(OUTPUT_DIR, 'hasil_pipeline.csv'), index=False)
display(df_pipeline)
print(f'\nRingkasan: {df_pipeline["legible"].sum()}/{len(df_pipeline)} video legible setelah '
      f'{LOOP_ATTACK[0]} CRF {LOOP_ATTACK[1]} (maks {EVAL_MAX_RETRY} percobaan; retry OFF). '
      f'Rata-rata percobaan: {df_pipeline["attempts"].mean():.2f}')

## 9. Evaluasi Lengkap (data uji, grup tidak pernah dilihat saat training)

1. **Imperceptibility** — PSNR & SSIM terhadap (a) video asli dan (b) output neural codec *tanpa* watermark. Nilai (b) mengukur distorsi yang murni disebabkan watermark; nilai (a) juga memuat distorsi codec itu sendiri. Kenaikan bitrate (bpp) akibat watermark ikut dilaporkan.
2. **Robustness** — BER tingkat video (soft voting antar frame) dan tingkat frame, serta *legible rate* (CRC valid + ID cocok) setelah: tanpa serangan, neural codec, H.264, HEVC, AV1 pada beberapa CRF, dan rescale 50%.
3. **False positive** — video asli tanpa watermark (juga setelah HEVC) diperiksa: berapa kali sistem keliru menyatakan ada watermark. Uji ini yang **tidak ada** di notebook lama, padahal paling penting untuk membuktikan decoder tidak sekadar menghafal.
4. **Baseline tradisional (DCT-QIM)** — watermark domain-DCT blok 8×8 pada kanal luma, payload sama, serangan sama. Kekuatannya **dikalibrasi otomatis** agar PSNR-nya setara dengan distorsi watermark usulan, sehingga perbandingan robustness adil (bukan baseline yang sengaja dibuat lemah). Ini mendukung klaim latar belakang bahwa watermark tradisional rusak oleh kompresi tinggi.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as sk_psnr, structural_similarity as sk_ssim

# ---------------- Baseline tradisional: DCT-QIM (luma, blok 8x8) ----------------
_D = cv2.dct(np.eye(8, dtype=np.float32))          # matriks DCT ortonormal 8x8
QIM_COEF = (1, 2)


def _blocks(Y):
    H8, W8 = Y.shape[0] // 8 * 8, Y.shape[1] // 8 * 8
    return Y[:H8, :W8].reshape(H8 // 8, 8, W8 // 8, 8).transpose(0, 2, 1, 3), (H8, W8)


def qim_embed(frames_u8, msg_bits, step=QIM_STEP):
    bits = np.asarray(msg_bits).astype(int)
    out = []
    for f in frames_u8:
        ycc = cv2.cvtColor(f, cv2.COLOR_RGB2YCrCb).astype(np.float32)
        B, (H8, W8) = _blocks(ycc[..., 0])
        C = _D @ B @ _D.T
        k = np.arange(B.shape[0] * B.shape[1]).reshape(B.shape[:2]) % len(bits)
        dither = (bits[k] * 2 - 1) * step / 4
        c = C[..., QIM_COEF[0], QIM_COEF[1]]
        C[..., QIM_COEF[0], QIM_COEF[1]] = step * np.round((c - dither) / step) + dither
        B2 = _D.T @ C @ _D
        ycc[:H8, :W8, 0] = B2.transpose(0, 2, 1, 3).reshape(H8, W8)
        out.append(cv2.cvtColor(np.clip(np.round(ycc), 0, 255).astype(np.uint8), cv2.COLOR_YCrCb2RGB))
    return np.stack(out)


def qim_soft(frames_u8, n_bits=MSG_BITS, step=QIM_STEP):
    '''Return soft value per frame per bit (positif -> bit 1), bentuk (T, n_bits).'''
    res = []
    for f in frames_u8:
        Y = cv2.cvtColor(f, cv2.COLOR_RGB2YCrCb).astype(np.float32)[..., 0]
        B, _ = _blocks(Y)
        c = (_D @ B @ _D.T)[..., QIM_COEF[0], QIM_COEF[1]].ravel()
        d = {}
        for b in (0, 1):
            dith = (b * 2 - 1) * step / 4
            d[b] = np.abs(c - (step * np.round((c - dith) / step) + dith))
        s = (d[0] - d[1]) / (step / 4)
        k = np.arange(len(c)) % n_bits
        res.append(np.bincount(k, weights=s, minlength=n_bits) / np.maximum(np.bincount(k, minlength=n_bits), 1))
    return np.stack(res)


def frame_metrics(a, b, every=4):
    idx = range(0, len(a), every)
    p = np.mean([sk_psnr(a[i], b[i], data_range=255) for i in idx])
    s = np.mean([sk_ssim(a[i], b[i], channel_axis=2, data_range=255) for i in idx])
    return float(p), float(s)


def attack_label(a):
    return a[0] if a[1] is None else f'{a[0]}_{a[1]}'


def calibrate_qim_step(frames_list, msg, target_psnr, lo=2.0, hi=80.0, iters=12):
    """Cari QIM_STEP agar PSNR baseline ~ PSNR distorsi watermark usulan (perbandingan adil)."""
    for _ in range(iters):
        mid = (lo + hi) / 2
        p = np.mean([frame_metrics(qim_embed(f, msg, mid), f, every=8)[0] for f in frames_list])
        lo, hi = (mid, hi) if p > target_psnr else (lo, mid)
    return (lo + hi) / 2


imp_rows, rob_rows, fp_rows = [], [], []
# 1) Imperceptibility metode usulan
for v, st in eval_store.items():
    rec, orig, wm, ref = st['record'], st['orig'], st['wm'], st['ref']
    p_wo, s_wo = frame_metrics(wm, orig); p_wr, s_wr = frame_metrics(wm, ref); p_ro, s_ro = frame_metrics(ref, orig)
    imp_rows.append({'video': v, 'PSNR_wm_vs_asli': p_wo, 'SSIM_wm_vs_asli': s_wo,
                     'PSNR_wm_vs_codec': p_wr, 'SSIM_wm_vs_codec': s_wr,
                     'PSNR_codec_vs_asli': p_ro, 'SSIM_codec_vs_asli': s_ro,
                     'bpp_codec': rec['bpp_codec'], 'bpp_wm': rec['bpp_wm']})

# 2) Kalibrasi baseline: distorsi DCT-QIM disamakan dengan distorsi watermark usulan
_target = float(np.mean([r['PSNR_wm_vs_codec'] for r in imp_rows]))
_cal_frames = [st['orig'][:24] for st in list(eval_store.values())[:3]]
QIM_STEP_CAL = calibrate_qim_step(_cal_frames, build_message(1).astype(int), _target)
print(f'Kalibrasi baseline: target PSNR {_target:.2f} dB -> QIM_STEP = {QIM_STEP_CAL:.2f}')

# 3) Robustness + false positive
for (v, st), imp in zip(tqdm(list(eval_store.items()), desc='Evaluasi'), imp_rows):
    rec, orig, wm = st['record'], st['orig'], st['wm']
    fps, wm_id = rec['fps'], int(rec['wm_id'], 16)
    msg = build_message(wm_id).astype(int)
    bl = qim_embed(orig, msg, QIM_STEP_CAL)
    imp['PSNR_baseline_vs_asli'], imp['SSIM_baseline_vs_asli'] = frame_metrics(bl, orig)

    for atk in EVAL_ATTACKS:
        if atk[0] == 'av1' and AV1_ENCODER is None:
            continue
        for method, src in [('Latent Neural Codec (usulan)', wm), ('DCT-QIM (tradisional)', bl)]:
            att, bpp = apply_attack(src, fps, atk)
            soft = extract_logits(att) if method.startswith('Latent') else qim_soft(att, step=QIM_STEP_CAL)
            dec = decode_message(soft.mean(0))
            rob_rows.append({'video': v, 'metode': method, 'serangan': attack_label(atk), 'bpp_file': bpp,
                             'BER_video': bit_error_rate((soft.mean(0) > 0).astype(int), msg),
                             'BER_frame': float(np.mean((soft > 0).astype(int) != msg[None])),
                             'legible': bool(dec['crc_ok'] and dec['id'] == wm_id)})

    # False positive: video ASLI tanpa watermark
    for kondisi, frames_neg in [('asli', orig), ('asli + HEVC CRF35', apply_attack(orig, fps, ('libx265', 35))[0])]:
        dec = decode_message(extract_logits(frames_neg).mean(0))
        best_p = 1.0
        for r in registry.values():
            ref_bits = np.array(list(r['payload_bits']), dtype=int)
            best_p = min(best_p, match_pvalue(int((dec['bits'] == ref_bits[:PAYLOAD_BITS]).sum()), PAYLOAD_BITS))
        fp_rows.append({'video': v, 'kondisi': kondisi, 'crc_ok': dec['crc_ok'],
                        'id_terdaftar': dec['crc_ok'] and dec['id_hex'] in registry,
                        'p_value_terbaik': best_p, 'deteksi_statistik': best_p < DETECT_PVALUE})

df_imp, df_rob, df_fp = pd.DataFrame(imp_rows), pd.DataFrame(rob_rows), pd.DataFrame(fp_rows)
for name, df in [('eval_imperceptibility', df_imp), ('eval_robustness', df_rob), ('eval_false_positive', df_fp)]:
    df.to_csv(os.path.join(OUTPUT_DIR, f'{name}.csv'), index=False)

print('=== IMPERCEPTIBILITY (rata-rata) ===')
display(df_imp.drop(columns='video').mean().round(4).to_frame('rata-rata'))

order = [attack_label(a) for a in EVAL_ATTACKS if not (a[0] == 'av1' and AV1_ENCODER is None)]
summary = (df_rob.groupby(['serangan', 'metode'])
           .agg(BER_video=('BER_video', 'mean'), BER_frame=('BER_frame', 'mean'),
                legible_rate=('legible', 'mean'), bpp_file=('bpp_file', 'mean'))
           .reset_index())
summary['serangan'] = pd.Categorical(summary['serangan'], order, ordered=True)
summary = summary.sort_values(['serangan', 'metode'])
summary.to_csv(os.path.join(OUTPUT_DIR, 'eval_robustness_ringkas.csv'), index=False)
print('\n=== ROBUSTNESS (rata-rata per serangan) ===')
display(summary.round(4))

print('\n=== FALSE POSITIVE (video tanpa watermark) ===')
display(df_fp.groupby('kondisi')[['crc_ok', 'id_terdaftar', 'deteksi_statistik']].mean())

# ---- Grafik ----
piv = summary.pivot(index='serangan', columns='metode', values='BER_frame').loc[order]
piv.plot(kind='bar', figsize=(12, 4)); plt.ylabel('BER tingkat frame'); plt.axhline(0.5, ls=':', c='gray')
plt.title('Robustness: BER frame-level setelah serangan (lebih rendah lebih baik)'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_ber_frame.png'), dpi=130); plt.show()

piv2 = summary.pivot(index='serangan', columns='metode', values='legible_rate').loc[order]
piv2.plot(kind='bar', figsize=(12, 4)); plt.ylabel('Legible rate (CRC & ID cocok)'); plt.ylim(0, 1.05)
plt.title('Tingkat keberhasilan ekstraksi payload per video'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eval_legible_rate.png'), dpi=130); plt.show()

# ---- Contoh visual ----
_v0 = next(iter(eval_store)); _s = eval_store[_v0]; _i = len(_s['wm']) // 2
diff = np.clip(np.abs(_s['wm'][_i].astype(int) - _s['ref'][_i].astype(int)) * 10, 0, 255).astype(np.uint8)
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
for a, img, t in zip(ax, [_s['orig'][_i], _s['ref'][_i], _s['wm'][_i], diff],
                     ['Asli', 'Neural codec tanpa WM', 'Neural codec + WM (latent)', '|WM − codec| × 10']):
    a.imshow(img); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'contoh_visual.png'), dpi=130); plt.show()

## 10. Output 2 — Sistem Verifikasi Otomatis & Validasi Integritas (Novelty 3)

`verify_video(path)` menjalankan alur **Upload → Ekstraksi → Validasi integritas → Laporan otomatis**:

| Tahap | Pemeriksaan | Dasar keputusan |
|---|---|---|
| Ekstraksi | payload dari seluruh frame (soft voting) | CRC-16 valid |
| Kepemilikan | ID dicari di registry; bila CRC gagal, dicari payload terdekat | *p-value* binomial < `DETECT_PVALUE` |
| Integritas file | SHA-256 dibandingkan dengan file yang terdaftar | identik bit-per-bit atau tidak |
| Integritas temporal | watermark dicek per segmen `SEGMENT_LEN` frame | akurasi bit segmen ≥ `SEG_BITACC_THRESH` |
| Integritas konten | pHash per frame dibandingkan dengan registry, setelah penyelarasan offset (tahan pemotongan awal) | jarak Hamming ≤ `PHASH_THRESH` |
| Kelengkapan | jumlah frame dibanding catatan | terpotong / ditambah |

Watermark bersifat **robust** (bertahan terhadap kompresi) sehingga membuktikan asal-usul, sedangkan pHash + pemeriksaan per segmen berfungsi sebagai lapisan **integritas** yang menunjukkan bagian mana yang diubah. Hasilnya disimpan sebagai JSON dan HTML di `OUTPUT_DIR/reports`.

Kemungkinan vonis: `AUTENTIK – IDENTIK`, `AUTENTIK – TERKOMPRESI ULANG, KONTEN UTUH`, `AUTENTIK – TERPOTONG`, `TERINDIKASI DIMANIPULASI`, `WATERMARK LEMAH / TIDAK PASTI`, `TIDAK ADA WATERMARK TERDAFTAR`.

In [ ]:
def align_offset(ref_h, q_h):
    '''Cari offset terbaik: frame q[i] dipasangkan dengan ref[i + off].'''
    n_r, n_q = len(ref_h), len(q_h)
    min_overlap = max(3, min(n_r, n_q) // 3)
    best_off, best_d = 0, float('inf')
    for off in range(-(n_q - 1), n_r):
        i0, i1 = max(0, -off), min(n_q, n_r - off)
        if i1 - i0 < min_overlap:
            continue
        idx = range(i0, i1, max(1, (i1 - i0) // 40))
        d = np.mean([hamming64(q_h[i], ref_h[i + off]) for i in idx])
        if d < best_d:
            best_off, best_d = off, d
    return best_off, best_d


def _verdict_color(v):
    return '#1e8e3e' if v.startswith('AUTENTIK') else ('#d93025' if 'MANIPULASI' in v else '#e37400')


def _report_plot_b64(frame_acc, dists, segments, fps):
    fig, ax = plt.subplots(2, 1, figsize=(10, 4.2), sharex=True)
    t = np.arange(len(frame_acc)) / fps
    ax[0].plot(t, frame_acc, lw=1); ax[0].axhline(SEG_BITACC_THRESH, ls='--', c='g')
    ax[0].set_ylabel('akurasi bit\nper frame'); ax[0].set_ylim(0.3, 1.02)
    d = np.array([np.nan if x is None else x for x in dists], dtype=float)
    ax[1].plot(t, d, lw=1, c='tab:orange'); ax[1].axhline(PHASH_THRESH, ls='--', c='r')
    ax[1].set_ylabel('jarak pHash'); ax[1].set_xlabel('detik')
    for s in segments:
        if not s['ok']:
            for a in ax:
                a.axvspan(s['start_s'], s['end_s'], color='red', alpha=0.15)
    plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=100); plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()


def _to_builtin(o):
    if isinstance(o, dict):
        return {k: _to_builtin(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_to_builtin(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return o


def render_html(rep, plot_b64=None):
    c = _verdict_color(rep['verdict'])
    rows = ''.join(f'<tr><th>{k}</th><td>{v}</td></tr>' for k, v in [
        ('File', rep['file']), ('Waktu verifikasi', rep['verified_at']),
        ('Resolusi / fps / codec', f"{rep['input']['width']}×{rep['input']['height']} / {rep['input']['fps']:.2f} / {rep['input']['codec']}"),
        ('Frame dianalisis', rep['input']['frames_analyzed']),
        ('ID terbaca', rep['extraction']['id_hex']), ('CRC valid', rep['extraction']['crc_ok']),
        ('Confidence', f"{rep['extraction']['confidence']:.3f}"),
        ('Metode identifikasi', rep['ownership'].get('method')), ('Pemilik', rep['ownership'].get('owner')),
        ('Video terdaftar', rep['ownership'].get('video_name')), ('Didaftarkan', rep['ownership'].get('created_at')),
        ('Bit cocok / p-value', f"{rep['ownership'].get('n_match')}/{PAYLOAD_BITS} / {rep['ownership'].get('p_value')}"),
        ('SHA-256 identik', rep['integrity'].get('sha256_match')),
        ('Frame tercatat / cocok', f"{rep['integrity'].get('frames_registered')} / {rep['integrity'].get('frames_matched')}"),
        ('Offset penyelarasan', rep['integrity'].get('offset')),
        ('Waktu proses', f"{rep['elapsed_s']:.1f} s")])
    seg_rows = ''.join(
        f"<tr style='background:{'#fff' if s['ok'] else '#fde7e7'}'><td>{s['index']}</td>"
        f"<td>{s['start_s']:.2f}–{s['end_s']:.2f}</td><td>{s['wm_bit_acc']:.3f}</td>"
        f"<td>{s['phash_median'] if s['phash_median'] is not None else '-'}</td>"
        f"<td>{'OK' if s['ok'] else ', '.join(s['issues'])}</td></tr>"
        for s in rep['integrity'].get('segments', []))
    img = f"<img style='max-width:100%' src='data:image/png;base64,{plot_b64}'/>" if plot_b64 else ''
    notes = ''.join(f'<li>{n}</li>' for n in rep['notes'])
    return f"""<html><head><meta charset='utf-8'><title>Laporan Verifikasi</title>
<style>body{{font-family:Arial,sans-serif;max-width:900px;margin:24px auto;color:#222}}
table{{border-collapse:collapse;width:100%;margin:10px 0}}th,td{{border:1px solid #ddd;padding:6px;text-align:left;font-size:13px}}
th{{background:#f5f5f5;width:35%}}.v{{padding:14px;border-radius:8px;color:#fff;font-size:20px;background:{c}}}</style></head>
<body><h2>Laporan Verifikasi Watermark &amp; Integritas Video</h2><div class='v'>{rep['verdict']}</div>
<ul>{notes}</ul><table>{rows}</table>{img}
<h3>Analisis per segmen</h3><table><tr><th>#</th><th>Waktu (s)</th><th>Akurasi bit WM</th><th>Median pHash</th><th>Status</th></tr>{seg_rows}</table>
</body></html>"""


def verify_video(path, reg=None, save=True):
    '''Output 2: Upload -> ekstraksi -> validasi integritas -> laporan otomatis.'''
    t0 = time.time()
    reg = load_registry() if reg is None else reg
    info = probe_video(path)
    frames, fps, _ = read_video(path, max_frames=MAX_VERIFY_FRAMES)
    logits = extract_logits(frames)
    dec = decode_message(logits.mean(0))
    rep = {'file': os.path.basename(path), 'verified_at': datetime.datetime.now().isoformat(timespec='seconds'),
           'input': {**info, 'frames_analyzed': len(frames)},
           'extraction': {'id_hex': dec['id_hex'], 'crc_ok': dec['crc_ok'], 'confidence': dec['confidence'],
                          'payload_bits': ''.join(map(str, dec['bits']))},
           'ownership': {}, 'integrity': {}, 'notes': []}

    # ---- Kepemilikan ----
    record, method = None, None
    best = None
    for r in reg.values():
        rb = np.array(list(r['payload_bits'][:PAYLOAD_BITS]), dtype=int)
        m = int((dec['bits'] == rb).sum())
        if best is None or m > best[0]:
            best = (m, r)
    if dec['crc_ok'] and dec['id_hex'] in reg:
        record, method = reg[dec['id_hex']], 'CRC valid + ID ada di registry'
        n_match = PAYLOAD_BITS
    elif best is not None and match_pvalue(best[0], PAYLOAD_BITS) < DETECT_PVALUE:
        record, method, n_match = best[1], 'deteksi statistik (CRC gagal, payload paling mirip)', best[0]
    else:
        n_match = best[0] if best else None
    p_val = match_pvalue(n_match, PAYLOAD_BITS) if n_match is not None else None
    rep['ownership'] = {'method': method, 'n_match': n_match, 'p_value': p_val}
    if dec['crc_ok'] and dec['id_hex'] not in reg:
        rep['notes'].append('CRC valid tetapi ID tidak ada di registry (registry lain / belum didaftarkan).')

    plot_b64 = None
    if record is None:
        weak = p_val is not None and p_val < 1e-3
        rep['verdict'] = 'WATERMARK LEMAH / TIDAK PASTI' if weak else 'TIDAK ADA WATERMARK TERDAFTAR'
        rep['notes'].append('Tidak ditemukan watermark yang cocok dengan registry.' if not weak else
                            'Ada kemiripan payload namun di bawah ambang deteksi; perlu pemeriksaan manual.')
    else:
        rep['ownership'].update({k: record[k] for k in ['wm_id', 'owner', 'video_name', 'created_at']})
        msg = np.array(list(record['payload_bits']), dtype=int)
        frame_acc = ((logits > 0).astype(int) == msg[None]).mean(1)

        ref_h = [int(h, 16) for h in record['phash']]
        q_h = phash_frames(frames)
        off, _ = align_offset(ref_h, q_h)
        dists = [hamming64(q_h[i], ref_h[i + off]) if 0 <= i + off < len(ref_h) else None for i in range(len(q_h))]
        matched = sum(d is not None and d <= PHASH_THRESH for d in dists)
        extra = sum(d is None for d in dists)

        segments = []
        for si, s in enumerate(range(0, len(frames), SEGMENT_LEN)):
            e = min(s + SEGMENT_LEN, len(frames))
            seg_acc = float(((logits[s:e].mean(0) > 0).astype(int) == msg).mean())
            ds = [d for d in dists[s:e] if d is not None]
            med = float(np.median(ds)) if ds else None
            issues = []
            if seg_acc < SEG_BITACC_THRESH:
                issues.append('watermark hilang/lemah')
            if med is None:
                issues.append('frame tidak ada di video asli')
            elif med > PHASH_THRESH:
                issues.append('konten berbeda dari asli')
            segments.append({'index': si, 'start_s': s / fps, 'end_s': e / fps, 'wm_bit_acc': seg_acc,
                             'phash_median': med, 'ok': not issues, 'issues': issues})

        sha_match = sha256_file(path) in record.get('sha256_known', [])
        bad = [s for s in segments if not s['ok']]
        n_reg = record['n_frames']
        trimmed = matched < n_reg - SEGMENT_LEN // 2 and len(frames) < MAX_VERIFY_FRAMES
        rep['integrity'] = {'sha256_match': sha_match, 'frames_registered': n_reg, 'frames_matched': matched,
                            'frames_extra': extra, 'offset': off, 'segments': segments,
                            'bad_segments': [s['index'] for s in bad]}
        if sha_match:
            rep['verdict'] = 'AUTENTIK – IDENTIK DENGAN FILE TERDAFTAR'
        elif bad:
            rep['verdict'] = 'TERINDIKASI DIMANIPULASI'
            rep['notes'].append('Segmen bermasalah: ' + '; '.join(
                f"{s['start_s']:.1f}–{s['end_s']:.1f}s ({', '.join(s['issues'])})" for s in bad))
        elif trimmed or off != 0:
            rep['verdict'] = 'AUTENTIK – TERPOTONG / PANJANG BERUBAH'
            rep['notes'].append(f'Hanya {matched} dari {n_reg} frame terdaftar yang ditemukan (offset {off}).')
        else:
            rep['verdict'] = 'AUTENTIK – TERKOMPRESI ULANG, KONTEN UTUH'
        if not sha_match:
            rep['notes'].append('File tidak identik bit-per-bit (wajar setelah kompresi/konversi ulang).')
        plot_b64 = _report_plot_b64(frame_acc, dists, segments, fps)

    rep['elapsed_s'] = time.time() - t0
    rep = _to_builtin(rep)
    if save:
        stem = f"{os.path.splitext(rep['file'])[0]}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')}"
        rep['report_json'] = os.path.join(OUTPUT_DIR, 'reports', stem + '.json')
        rep['report_html'] = os.path.join(OUTPUT_DIR, 'reports', stem + '.html')
        with open(rep['report_json'], 'w') as f:
            json.dump(rep, f, indent=1)
        with open(rep['report_html'], 'w') as f:
            f.write(render_html(rep, plot_b64))
    rep['_html'] = render_html(rep, plot_b64)
    return rep


print('verify_video() siap.')

### 10.1 Uji skenario verifikasi (autentik, kompresi ulang, manipulasi, pemotongan, tanpa watermark)

Skenario ini menjadi bukti fungsional Output 2 untuk bab hasil. Tabel ringkasnya disimpan di `verifikasi_skenario.csv`.

In [ ]:
_legible = [v for v, s in eval_store.items() if s['record']['legible']]
if not _legible:
    print('Belum ada video legible — skenario verifikasi dilewati.')
else:
    v_demo = _legible[0]
    rec = eval_store[v_demo]['record']
    other = next((v for v in eval_store if v != v_demo), v_demo)
    wm_frames, fps, _ = read_video(rec['watermarked_path'], max_frames=None)
    T = len(wm_frames)
    D = lambda n: os.path.join(OUTPUT_DIR, 'demo', n)
    a, b = T // 3, min(T, T // 3 + max(SEGMENT_LEN, T // 6))

    tamper = wm_frames.copy(); tamper[a:b] = eval_store[other]['orig'][a:b]
    boxed = wm_frames.copy()
    H, W = FRAME_SIZE
    boxed[a:b, H // 4:3 * H // 4, W // 4:3 * W // 4] = 0
    cut = max(SEGMENT_LEN, T // 4)

    scenarios = [
        ('1. File watermark asli (lossless)', rec['watermarked_path']),
        ('2. Hasil loop HEVC CRF35 (file distribusi)', rec['compressed_path']),
        ('3. Dikompres ulang AV1 CRF45' if AV1_ENCODER else '3. Dikompres ulang H.264 CRF40',
         write_video(wm_frames, D('s3' + codec_ext('av1' if AV1_ENCODER else 'libx264')), fps,
                     codec='av1' if AV1_ENCODER else 'libx264', crf=45 if AV1_ENCODER else 40)),
        ('4. Segmen diganti video lain + H.264 CRF28', write_video(tamper, D('s4_tamper.mp4'), fps, 'libx264', 28)),
        ('5. Area tengah ditutup + HEVC CRF30', write_video(boxed, D('s5_tamper_box.mp4'), fps, 'libx265', 30)),
        (f'6. {cut} frame awal dipotong + HEVC CRF35', write_video(wm_frames[cut:], D('s6_trim.mp4'), fps, 'libx265', 35)),
        ('7. Video asli TANPA watermark', os.path.join(DATASET_DIR, v_demo)),
    ]
    rows = []
    for name, p in scenarios:
        r = verify_video(p)
        rows.append({'skenario': name, 'vonis': r['verdict'], 'ID': r['extraction']['id_hex'],
                     'CRC': r['extraction']['crc_ok'], 'metode': r['ownership'].get('method'),
                     'p_value': r['ownership'].get('p_value'),
                     'segmen_bermasalah': r['integrity'].get('bad_segments'), 'laporan': r.get('report_html')})
    df_ver = pd.DataFrame(rows)
    df_ver.to_csv(os.path.join(OUTPUT_DIR, 'verifikasi_skenario.csv'), index=False)
    display(df_ver.drop(columns='laporan'))

    if IN_COLAB:
        from IPython.display import HTML
        display(HTML(verify_video(scenarios[3][1], save=False)['_html']))

## 11. Tool Verifikasi Interaktif (Gradio)

Antarmuka web sederhana: unggah video → vonis + laporan HTML + file JSON yang bisa diunduh. Di Colab, `share=True` menghasilkan tautan publik sementara yang bisa dipakai untuk demo sidang. Model dan registry yang dipakai adalah yang sudah dimuat di sesi ini.

In [ ]:
if RUN_GRADIO:
    if importlib.util.find_spec('gradio') is None:
        _pip('gradio')
    import gradio as gr

    def gr_verify(file_path):
        if not file_path:
            return 'Silakan unggah video.', '', None
        r = verify_video(file_path)
        md = (f"## {r['verdict']}\n\n**ID:** `{r['extraction']['id_hex']}` · **CRC:** {r['extraction']['crc_ok']} · "
              f"**Pemilik:** {r['ownership'].get('owner', '-')} · **Waktu proses:** {r['elapsed_s']:.1f} s")
        return md, r['_html'], r.get('report_json')

    _major = int(gr.__version__.split('.')[0])
    _flag = {'flagging_mode': 'never'} if _major >= 5 else {'allow_flagging': 'never'}
    demo = gr.Interface(
        fn=gr_verify,
        inputs=gr.File(label='Unggah video (.mp4 / .mkv / .avi / .mov)', type='filepath'),
        outputs=[gr.Markdown(), gr.HTML(), gr.File(label='Laporan JSON')],
        title='Verifikasi Watermark & Integritas Video (Latent Neural Codec)',
        description='Upload video → ekstraksi watermark → validasi integritas → laporan otomatis.',
        **_flag,
    )
    demo.launch(share=IN_COLAB, debug=False)

## 12. Pemetaan ke Tujuan Riset & Catatan untuk Penulisan

**Output 1 — algoritma watermarking berbasis neural codec:** `embed_video()` + `LatentWatermarkEmbedder` + `LatentWatermarkExtractor`, dengan loop flowchart di `run_watermark_pipeline()`. Bukti: `eval_imperceptibility.csv`, `eval_robustness_ringkas.csv`, `eval_ber_frame.png`, `eval_legible_rate.png`, `contoh_visual.png`.

**Output 2 — tool verifikasi + validasi integritas:** `verify_video()` + registry + laporan JSON/HTML + antarmuka Gradio. Bukti: `verifikasi_skenario.csv` dan folder `reports/`.

**Tabel yang sebaiknya masuk bab hasil:**
1. Imperceptibility: PSNR/SSIM (WM vs asli, WM vs codec, codec vs asli, baseline vs asli), bpp codec vs bpp WM.
2. Robustness per serangan dan CRF: BER video, BER frame, legible rate — usulan vs DCT-QIM.
3. False positive pada video tanpa watermark.
4. Skenario verifikasi dan vonisnya.

**Keterbatasan yang perlu ditulis jujur:**
- Neural codec yang dipakai adalah *learned image codec* per frame (intra), belum *neural video codec* dengan prediksi antar-frame (misalnya DCVC atau `ssf2020` di CompressAI). Ini arah pengembangan yang wajar.
- Resolusi diproses pada `FRAME_SIZE`; video beresolusi lain di-resize saat verifikasi.
- Integritas berbasis pHash mendeteksi perubahan konten yang cukup besar; perubahan sangat kecil (misalnya mengubah beberapa piksel) bisa lolos. Ambang `PHASH_THRESH` sebaiknya dikalibrasi dari data (bandingkan distribusi jarak video autentik terkompresi vs video yang dimanipulasi).
- Keamanan bergantung pada kerahasiaan `SECRET_KEY` dan integritas file registry.